# Часть II. Бустинги XGBoost и CatBoost

Жунёв Андрей Александрович РИМ-150950

**Цель:** Продемонстрировать работу алгоритмов градиентного бустинга XGBoost и CatBoost на примере датасета предсказания оценки вина (решение задачи регрессии).

**Набор данных:** [Wine Quality (Red Wine Dataset)](https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009)
**Источник:** Kaggle / UCI Machine Learning Repository  
**Размер:** 1599 образцов × 12 признаков  
**Целевая переменная:** quality (оценка 3-8 баллов)

**Примечание:** Детальный EDA и обоснование предобработки данных выполнены в части 1 (`part_1_sklearn.ipynb`).


## Оглавление

1. [I. Загрузка и подготовка данных](#I.-Загрузка-и-подготовка-данных)
2. [II. Предварительная обработка данных](#II.-Предварительная-обработка-данных)
   - [1. Pipeline](#1.-Pipeline)
   - [2. Создание расширенного датасета с синтетическими признаками](#2.-Создание-расширенного-датасета-с-синтетическими-признаками)
3. [III. Обучение базовых моделей](#III.-Обучение-базовых-моделей)
   - [1. Метрики для сравнения моделей](#1.-Метрики-для-сравнения-моделей)
   - [2. Выбор моделей](#2.-Выбор-моделей)
   - [3. Обучение XGBoost (базовая модель)](#3.-Обучение-XGBoost-(базовая-модель))
   - [4. Обучение CatBoost (базовая модель)](#4.-Обучение-CatBoost-(базовая-модель))
   - [5. Сравнение базовых моделей](#5.-Сравнение-базовых-моделей)
4. [IV. Оптимизация выбранных моделей](#IV.-Оптимизация-выбранных-моделей)
   - [1. Оптимизация XGBoost](#1.-Оптимизация-XGBoost)
   - [2. Оптимизация CatBoost](#2.-Оптимизация-CatBoost)
5. [V. Ансамбль XGBoost + CatBoost (VotingRegressor)](#V.-Ансамбль-XGBoost-+-CatBoost-(VotingRegressor))
6. [VI. Итоговое сравнение моделей](#VI.-Итоговое-сравнение-моделей)
   - [1. Общие итоги и выбор лучших](#1.-Общие-итоги-и-выбор-лучших)
   - [2. Анализ распределения ошибок для лучших моделей](#2.-Анализ-распределения-ошибок-для-лучших-моделей)
7. [VII. Выводы](#VII.-Выводы)


## Импорты


In [1]:
# ============================================================================
# ИМПОРТЫ НЕОБХОДИМЫХ БИБЛИОТЕК
# ============================================================================

# Базовые библиотеки для работы с данными
import pandas as pd
import numpy as np
from pathlib import Path

# Визуализация (Plotly для интерактивных графиков)
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Scikit-learn - разделение данных
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    ShuffleSplit,
    cross_validate,
)

# Scikit-learn - предобработка
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Модели XGBoost и CatBoost
import xgboost as xgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# Scikit-learn - метрики
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
)

# Утилиты
import time
import warnings
warnings.filterwarnings('ignore')

# Настройка отображения pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print("✅ Все библиотеки успешно импортированы")

# ============================================================================
# Глобальный список для хранения результатов всех моделей
# ============================================================================
MODEL_RESULTS = []

def add_result(model_name, features_desc, model, X_train, y_train, X_test, y_test):
    """Добавляет результаты модели в глобальный список MODEL_RESULTS.
    
    Параметры:
    ----------
    model_name : str
        Название модели для отображения
    features_desc : str
        Описание используемых признаков
    model : estimator
        Обученная модель
    X_train : DataFrame
        Тренировочные признаки
    y_train : Series
        Тренировочные метки
    X_test : DataFrame
        Тестовые признаки
    y_test : Series
        Тестовые метки
    """
    # Предикты
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Метрики
    r2_train = r2_score(y_train, y_train_pred)
    r2_test = r2_score(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    mae = mean_absolute_error(y_test, y_test_pred)
    
    # Сохраняем
    MODEL_RESULTS.append({
        'Model': model_name,
        'Features': features_desc,
        'R² Train': r2_train,
        'R² Test': r2_test,
        'Gap': r2_train - r2_test,
        'RMSE': rmse,
        'MAE': mae
    })
    
print("✅ Функция add_result() создана")


✅ Все библиотеки успешно импортированы
✅ Функция add_result() создана


In [2]:
def calculate_metrics(y_true, y_pred, model_name="Model", y_train_true=None, y_train_pred=None):
    """Расчет и вывод метрик для регрессии
    
    Параметры:
    ----------
    y_true : array-like
        Истинные значения для тестовой выборки
    y_pred : array-like
        Предсказанные значения для тестовой выборки
    model_name : str
        Имя модели для вывода
    y_train_true : array-like, optional
        Истинные значения для тренировочной выборки (для сравнения)
    y_train_pred : array-like, optional
        Предсказанные значения для тренировочной выборки (для сравнения)
    """
    # Метрики на тестовой выборке
    r2_test = r2_score(y_true, y_pred)
    rmse_test = np.sqrt(mean_squared_error(y_true, y_pred))
    mae_test = mean_absolute_error(y_true, y_pred)
    
    print(f"\n{model_name}:")
    print(f"{'Метрика':<20} {'Тренировочный набор':<25} {'Тестовый набор':<20}")
    print("-" * 65)
    
    # Если переданы train данные, вычисляем метрики и для них
    if y_train_true is not None and y_train_pred is not None:
        r2_train = r2_score(y_train_true, y_train_pred)
        rmse_train = np.sqrt(mean_squared_error(y_train_true, y_train_pred))
        mae_train = mean_absolute_error(y_train_true, y_train_pred)
        
        print(f"{'R²':<20} {r2_train:<25.4f} {r2_test:<20.4f}")
        print(f"{'RMSE':<20} {rmse_train:<25.4f} {rmse_test:<20.4f}")
        print(f"{'MAE':<20} {mae_train:<25.4f} {mae_test:<20.4f}")
        
        # Разница между train и test R² (для оценки переобучения)
        r2_diff = r2_train - r2_test
        print(f"\nРазница R² (train - test): {r2_diff:.4f}")
        if r2_diff > 0.1:
            print("⚠️ Возможное переобучение (разница > 0.1)")
        else:
            print("✅ Переобучение незначительное (разница <= 0.1)")
        
        return {
            'train': {'r2': r2_train, 'rmse': rmse_train, 'mae': mae_train},
            'test': {'r2': r2_test, 'rmse': rmse_test, 'mae': mae_test},
            'r2_diff': r2_diff
        }
    else:
        # Если train данные не переданы, выводим только test метрики
        print(f"{'R²':<20} {'N/A':<25} {r2_test:<20.4f}")
        print(f"{'RMSE':<20} {'N/A':<25} {rmse_test:<20.4f}")
        print(f"{'MAE':<20} {'N/A':<25} {mae_test:<20.4f}")
        
        return {'r2': r2_test, 'rmse': rmse_test, 'mae': mae_test}

print("✅ Функция calculate_metrics() создана")


✅ Функция calculate_metrics() создана


In [3]:
def plot_model_report_separate(
    metrics: dict,
    feature_importance: pd.DataFrame | None,
    model_name: str,
    width_metrics: int = 800,
    height_metrics: int = 500,
    width_fi: int = 800,
    height_fi: int = 600,
):
    """Построение двух отдельных графиков: метрики train/test и важность признаков.

    Параметры
    ----------
    metrics : dict
        Словарь метрик вида:
        {
            'train': {'r2': float, 'rmse': float, 'mae': float},
            'test':  {'r2': float, 'rmse': float, 'mae': float}
        }.

    feature_importance : pandas.DataFrame | None
        Таблица важностей признаков с колонками ['feature', 'importance'].

    model_name : str
        Название модели для заголовков графиков.
    """
    # Метрики (отдельный Figure)
    metrics_df = pd.DataFrame({
        'Метрика': ['R²', 'RMSE', 'MAE'],
        'Train': [metrics['train']['r2'], metrics['train']['rmse'], metrics['train']['mae']],
        'Test':  [metrics['test']['r2'],  metrics['test']['rmse'],  metrics['test']['mae']],
    })

    fig_metrics = go.Figure()
    fig_metrics.add_trace(go.Bar(
        name='Train',
        x=metrics_df['Метрика'],
        y=metrics_df['Train'],
        marker_color='steelblue',
        text=[f'{v:.4f}' for v in metrics_df['Train']],
        textposition='auto'
    ))
    fig_metrics.add_trace(go.Bar(
        name='Test',
        x=metrics_df['Метрика'],
        y=metrics_df['Test'],
        marker_color='crimson',
        text=[f'{v:.4f}' for v in metrics_df['Test']],
        textposition='auto'
    ))
    fig_metrics.update_layout(
        title={'text': f'Метрики качества: {model_name}', 'x': 0.5, 'xanchor': 'center'},
        xaxis_title='Метрика',
        yaxis_title='Значение',
        barmode='group',
        template='plotly_white',
        width=width_metrics,
        height=height_metrics,
        legend_title_text="Split"
    )

    fig_fi = None
    if feature_importance is not None and len(feature_importance) > 0:
        fi = feature_importance.copy().sort_values("importance", ascending=True)
        fig_fi = go.Figure()
        fig_fi.add_trace(go.Bar(
            x=fi['importance'],
            y=fi['feature'],
            orientation='h',
            marker_color='coral'
        ))
        fig_fi.update_layout(
            title={'text': 'Важность признаков', 'x': 0.5, 'xanchor': 'center'},
            xaxis_title='Важность',
            yaxis_title='Признак',
            template='plotly_white',
            width=width_fi,
            height=height_fi,
            showlegend=False
        )

    fig_metrics.show()
    if fig_fi is not None:
        fig_fi.show()

    return None

print("✅ Функция plot_model_report_separate() создана")


✅ Функция plot_model_report_separate() создана


In [4]:
# ============================================================================
# УТИЛИТЫ ДЛЯ FEATURE ENGINEERING
# ============================================================================

def add_wine_features(df):
    """Добавляет синтетические химические признаки к данным вина.
    
    Функция создает новые признаки на основе химических свойств вина,
    которые могут помочь модели лучше улавливать взаимосвязи.
    
    Параметры:
    ----------
    df : pd.DataFrame
        Исходный DataFrame с признаками вина
        
    Возвращает:
    ----------
    pd.DataFrame
        DataFrame с добавленными синтетическими признаками
        
    Новые признаки:
    - total_acidity: сумма всех кислот (fixed + volatile + citric)
    - vol_fixed_ratio: отношение летучей к постоянной кислотности
    - free_so2_ratio: процент свободного SO2 от общего
    - alc_density_prod: произведение алкоголя и плотности (тело вина)
    - sugar_alc_ratio: отношение сахара к алкоголю
    """
    data = df.copy()
    
    # Общая кислотность (сумма всех кислот)
    data['total_acidity'] = data['fixed acidity'] + data['volatile acidity'] + data['citric acid']
    
    # Отношение летучей кислотности к постоянной (маркер качества)
    data['vol_fixed_ratio'] = data['volatile acidity'] / (data['fixed acidity'] + 1e-5)
    
    # Процент свободного SO2 от общего (антиокислительная защита)
    data['free_so2_ratio'] = data['free sulfur dioxide'] / (data['total sulfur dioxide'] + 1e-5)
    
    # Произведение алкоголя и плотности (индикатор тела вина)
    data['alc_density_prod'] = data['alcohol'] * data['density']
    
    # Отношение сахара к алкоголю (баланс сладости и крепости)
    data['sugar_alc_ratio'] = data['residual sugar'] / (data['alcohol'] + 1e-5)
    
    return data

# Список имен новых признаков для использования в обоих алгоритмах
NEW_FEATURE_NAMES = ['total_acidity', 'vol_fixed_ratio', 'free_so2_ratio', 
                     'alc_density_prod', 'sugar_alc_ratio']

print("✅ Функция add_wine_features() создана")
print(f"✅ Новые признаки: {NEW_FEATURE_NAMES}")


✅ Функция add_wine_features() создана
✅ Новые признаки: ['total_acidity', 'vol_fixed_ratio', 'free_so2_ratio', 'alc_density_prod', 'sugar_alc_ratio']


# I. Загрузка и подготовка данных

Детальный анализ данных (EDA) был выполнен в работе `part_1_sklearn.ipynb`. 

**Ключевые выводы из предыдущего EDA:**
- Все 11 признаков числовые, пропусков нет
- Целевая переменная `quality` принимает значения от 3 до 8
- Сильные связи с целевой: alcohol (≈0.48), volatile acidity (≈-0.39), sulphates (≈0.25)
- Признаки демонстрируют асимметрию и наличие выбросов
- Требуется предобработка: стандартизация и преобразование признаков


In [5]:
# ============================================================================
# ЗАГРУЗКА ДАННЫХ
# ============================================================================

data_path = Path('data/wine-quality/winequality-red.csv')
df = pd.read_csv(data_path)

print(f"Размерность: {df.shape}")
print(f"Признаки: {list(df.columns)}")
print("Целевая переменная: quality")


Размерность: (1599, 12)
Признаки: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']
Целевая переменная: quality


In [6]:
# Предпросмотр данных
target_column = 'quality'
df.head()


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


# II. Предварительная обработка данных

Используем тот же pipeline предобработки, что и в части 1: PowerTransformer (Yeo-Johnson) с одновременной стандартизацией.


## 1. Pipeline

In [7]:
# Разделение данных на матрицу признаков X и целевую переменную y
X = df.drop(columns=[target_column])
y = df[target_column]

print(f"Количество признаков: {X.shape[1]}")
print(f"\nСписок признаков:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("\nРазделение данных на train/test:")
print("="*60)
print(f"Размер тренировочной выборки (X_train): {X_train.shape}")
print(f"Размер тестовой выборки (X_test): {X_test.shape}")
print(f"\nСоотношение train/test: {X_train.shape[0]}/{X_test.shape[0]} = {X_train.shape[0]/X_test.shape[0]:.2f}:1")


Количество признаков: 11

Список признаков:
   1. fixed acidity
   2. volatile acidity
   3. citric acid
   4. residual sugar
   5. chlorides
   6. free sulfur dioxide
   7. total sulfur dioxide
   8. density
   9. pH
  10. sulphates
  11. alcohol

Разделение данных на train/test:
Размер тренировочной выборки (X_train): (1279, 11)
Размер тестовой выборки (X_test): (320, 11)

Соотношение train/test: 1279/320 = 4.00:1


In [8]:
# Построение Pipeline с использованием PowerTransformer (Yeo-Johnson)
print("="*60)
print("Построение Pipeline предобработки:")
print("="*60)

# Все числовые признаки подвергаются Yeo-Johnson с одновременной стандартизацией
feature_order = list(X_train.columns)

# Создание ColumnTransformer: один шаг PowerTransformer для всех признаков
preprocessor = ColumnTransformer(
    transformers=[
        ('power_transform', PowerTransformer(method='yeo-johnson', standardize=True), feature_order)
    ],
    remainder='drop'
)

# Pipeline состоит только из предобработки
preprocessing_pipeline = Pipeline([
    ('preprocessor', preprocessor)
])

# Применение предобработки к тренировочным и тестовым данным
X_train_final = preprocessing_pipeline.fit_transform(X_train)
X_test_final = preprocessing_pipeline.transform(X_test)

# Преобразование обратно в DataFrame для удобства
X_train_final = pd.DataFrame(X_train_final, columns=feature_order, index=X_train.index)
X_test_final = pd.DataFrame(X_test_final, columns=feature_order, index=X_test.index)

print("Pipeline создан и применен (PowerTransformer, Yeo-Johnson)")
print(f"\nРазмерность данных после предобработки:")
print(f"  X_train_final: {X_train_final.shape}")
print(f"  X_test_final: {X_test_final.shape}")


Построение Pipeline предобработки:
Pipeline создан и применен (PowerTransformer, Yeo-Johnson)

Размерность данных после предобработки:
  X_train_final: (1279, 11)
  X_test_final: (320, 11)


## 2. Создание расширенного датасета с синтетическими признаками

Для экспериментов с Feature Engineering создадим второй вариант датасета с добавленными химическими признаками. Это позволит сравнить модели на базовых признаках (11) и расширенных (16) без дублирования кода в каждой секции.

**Новые признаки:**
- `total_acidity` — общая кислотность (fixed + volatile + citric)
- `vol_fixed_ratio` — отношение летучей к постоянной кислотности
- `free_so2_ratio` — процент свободного SO2
- `alc_density_prod` — произведение алкоголя и плотности
- `sugar_alc_ratio` — отношение сахара к алкоголю


In [9]:
# ============================================================================
# СОЗДАНИЕ РАСШИРЕННОГО ДАТАСЕТА С СИНТЕТИЧЕСКИМИ ПРИЗНАКАМИ
# ============================================================================

# 1. Применяем add_wine_features к исходным данным (до Pipeline)
X_train_with_new = add_wine_features(X_train)
X_test_with_new = add_wine_features(X_test)

# 2. Извлекаем только новые признаки
X_train_new_only = X_train_with_new[NEW_FEATURE_NAMES]
X_test_new_only = X_test_with_new[NEW_FEATURE_NAMES]

# 3. Применяем PowerTransformer к новым признакам
power_transformer_new = PowerTransformer(method='yeo-johnson', standardize=True)
X_train_new_transformed = power_transformer_new.fit_transform(X_train_new_only)
X_test_new_transformed = power_transformer_new.transform(X_test_new_only)

# Преобразуем в DataFrame
X_train_new_transformed = pd.DataFrame(
    X_train_new_transformed, 
    columns=NEW_FEATURE_NAMES, 
    index=X_train.index
)
X_test_new_transformed = pd.DataFrame(
    X_test_new_transformed, 
    columns=NEW_FEATURE_NAMES, 
    index=X_test.index
)

# 4. Объединяем базовые признаки с новыми
X_train_plus = pd.concat([X_train_final, X_train_new_transformed], axis=1)
X_test_plus = pd.concat([X_test_final, X_test_new_transformed], axis=1)

print("="*70)
print("ПОДГОТОВЛЕНЫ ДВА ВАРИАНТА ДАТАСЕТА:")
print("="*70)
print(f"\n1. Базовый датасет (X_train_final, X_test_final):")
print(f"   Количество признаков: {X_train_final.shape[1]}")
print(f"   Размер: {X_train_final.shape[0]} train / {X_test_final.shape[0]} test")

print(f"\n2. Расширенный датасет (X_train_plus, X_test_plus):")
print(f"   Количество признаков: {X_train_plus.shape[1]}")
print(f"   Размер: {X_train_plus.shape[0]} train / {X_test_plus.shape[0]} test")

print(f"\nДобавленные признаки: {NEW_FEATURE_NAMES}")


ПОДГОТОВЛЕНЫ ДВА ВАРИАНТА ДАТАСЕТА:

1. Базовый датасет (X_train_final, X_test_final):
   Количество признаков: 11
   Размер: 1279 train / 320 test

2. Расширенный датасет (X_train_plus, X_test_plus):
   Количество признаков: 16
   Размер: 1279 train / 320 test

Добавленные признаки: ['total_acidity', 'vol_fixed_ratio', 'free_so2_ratio', 'alc_density_prod', 'sugar_alc_ratio']


# III. Обучение базовых моделей

## 1. Метрики для сравнения моделей


**Выбранные метрики:**
- **R² (коэффициент детерминации)** — основная метрика
- **RMSE (Root Mean Squared Error)** — корень из среднеквадратичной ошибки
- **MAE (Mean Absolute Error)** — средняя абсолютная ошибка

**Обоснование:** те же метрики, что использовались в части 1 для обеспечения сравнимости результатов.

## 2. Выбор моделей



**Выбранные модели:**
1. **XGBoost (XGBRegressor)** — эффективный градиентный бустинг с регуляризацией
2. **CatBoost (CatBoostRegressor)** — градиентный бустинг с автоматической обработкой категориальных признаков

**Обоснование выбора:**

1. **XGBoost:**
   - Один из наиболее популярных алгоритмов градиентного бустинга
   - Встроенная регуляризация (L1 и L2)
   - Поддержка параллельного обучения
   - Эффективная обработка разреженных данных

2. **CatBoost:**
   - Разработан Яндексом, оптимизирован для категориальных признаков
   - Ordered Boosting для уменьшения переобучения
   - Symmetric Trees для быстрого инференса
   - Хорошо работает "из коробки" без тщательной настройки

## 3. Обучение XGBoost (базовая модель)

**Цель:** Посмотреть как модель работает "из коробки" с базовыми параметрами.


In [10]:
# Базовая модель XGBoost
xgb_baseline = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    objective='reg:squarederror'
)

# Обучение на тренировочной выборке
print("Обучение XGBoost...")
xgb_baseline.fit(X_train_final, y_train)

# Предсказания на тренировочной и тестовой выборках
y_train_pred_xgb = xgb_baseline.predict(X_train_final)
y_pred_xgb = xgb_baseline.predict(X_test_final)

# Метрики с сравнением train/test
metrics_xgb = calculate_metrics(
    y_test, y_pred_xgb, 
    "XGBoost (Baseline)",
    y_train_true=y_train,
    y_train_pred=y_train_pred_xgb
)

# Важность признаков
feature_importance_xgb = pd.DataFrame({
    'feature': X_train_final.columns,
    'importance': xgb_baseline.feature_importances_
}).sort_values('importance', ascending=False)

print("\nВажность признаков (топ-5):")
print(feature_importance_xgb.head())

# Добавляем результат в глобальный список
add_result("XGB Baseline", "All (11)", xgb_baseline, X_train_final, y_train, X_test_final, y_test)


Обучение XGBoost...

XGBoost (Baseline):
Метрика              Тренировочный набор       Тестовый набор      
-----------------------------------------------------------------
R²                   0.6080                    0.4280              
RMSE                 0.5050                    0.6114              
MAE                  0.3895                    0.4892              

Разница R² (train - test): 0.1800
⚠️ Возможное переобучение (разница > 0.1)

Важность признаков (топ-5):
                 feature  importance
10               alcohol    0.312575
9              sulphates    0.160252
1       volatile acidity    0.130139
6   total sulfur dioxide    0.063403
5    free sulfur dioxide    0.055129


In [11]:
plot_model_report_separate(metrics_xgb, feature_importance_xgb, model_name="XGBoost (Baseline)")


Вообще конечно я ожидал большего, даже интересно, почему на тесте результат такой невзрачный. Возможно, это связано с тем, что одеяло важности перетягивает на себя один признак - алкоголь.

## 4. Обучение CatBoost (базовая модель)

**Цель:** Посмотреть как модель работает "из коробки" с базовыми параметрами.


In [12]:
# Базовая модель CatBoost
cat_baseline = CatBoostRegressor(
    iterations=100,
    depth=3,
    learning_rate=0.1,
    random_state=42,
    verbose=False,
    loss_function='RMSE'
)

# Обучение на тренировочной выборке
print("Обучение CatBoost...")
cat_baseline.fit(X_train_final, y_train)

# Предсказания на тренировочной и тестовой выборках
y_train_pred_cat = cat_baseline.predict(X_train_final)
y_pred_cat = cat_baseline.predict(X_test_final)

# Метрики с сравнением train/test
metrics_cat = calculate_metrics(
    y_test, y_pred_cat, 
    "CatBoost (Baseline)",
    y_train_true=y_train,
    y_train_pred=y_train_pred_cat
)

# Важность признаков
feature_importance_cat = pd.DataFrame({
    'feature': X_train_final.columns,
    'importance': cat_baseline.feature_importances_
}).sort_values('importance', ascending=False)

print("\nВажность признаков (топ-5):")
print(feature_importance_cat.head())

# Добавляем результат в глобальный список
add_result("Cat Baseline", "All (11)", cat_baseline, X_train_final, y_train, X_test_final, y_test)


Обучение CatBoost...

CatBoost (Baseline):
Метрика              Тренировочный набор       Тестовый набор      
-----------------------------------------------------------------
R²                   0.4827                    0.4272              
RMSE                 0.5801                    0.6118              
MAE                  0.4512                    0.4925              

Разница R² (train - test): 0.0555
✅ Переобучение незначительное (разница <= 0.1)

Важность признаков (топ-5):
                 feature  importance
10               alcohol   35.291242
9              sulphates   25.501607
1       volatile acidity   15.206201
6   total sulfur dioxide    7.235130
8                     pH    4.952213


In [13]:
plot_model_report_separate(metrics_cat, feature_importance_cat, model_name="CatBoost (Baseline)")


У кэтбуста в базе все отлично с переобучением - его почти нет, что великолепно на самом деле. К тому же, теперь одеяло важности признаков перетягивают на себя 3 признака, а не только алкоголь, что конечно логично но все равно. При этом напрягает низкая точность.

## 5. Сравнение базовых моделей


In [14]:
# Сравнение базовых моделей
comparison_baseline = pd.DataFrame({
    'Model': ['XGBoost', 'CatBoost'],
    'R²': [
        r2_score(y_test, y_pred_xgb),
        r2_score(y_test, y_pred_cat)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, y_pred_xgb)),
        np.sqrt(mean_squared_error(y_test, y_pred_cat))
    ],
    'MAE': [
        mean_absolute_error(y_test, y_pred_xgb),
        mean_absolute_error(y_test, y_pred_cat)
    ]
})

print("="*60)
print("Сравнение базовых моделей на тестовой выборке:")
print("="*60)
print(comparison_baseline.to_string(index=False))


Сравнение базовых моделей на тестовой выборке:
   Model       R²     RMSE      MAE
 XGBoost 0.428030 0.611380 0.489195
CatBoost 0.427181 0.611834 0.492519


In [15]:
# Визуализация сравнения метрик
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('R² Score', 'RMSE', 'MAE'),
    horizontal_spacing=0.1
)

# R² Score
fig.add_trace(
    go.Bar(
        x=comparison_baseline['Model'],
        y=comparison_baseline['R²'],
        name='R²',
        marker_color='steelblue',
        text=[f'{val:.4f}' for val in comparison_baseline['R²']],
        textposition='auto'
    ),
    row=1, col=1
)

# RMSE
fig.add_trace(
    go.Bar(
        x=comparison_baseline['Model'],
        y=comparison_baseline['RMSE'],
        name='RMSE',
        marker_color='coral',
        text=[f'{val:.4f}' for val in comparison_baseline['RMSE']],
        textposition='auto',
        showlegend=False
    ),
    row=1, col=2
)

# MAE
fig.add_trace(
    go.Bar(
        x=comparison_baseline['Model'],
        y=comparison_baseline['MAE'],
        name='MAE',
        marker_color='lightgreen',
        text=[f'{val:.4f}' for val in comparison_baseline['MAE']],
        textposition='auto',
        showlegend=False
    ),
    row=1, col=3
)

fig.update_layout(
    title_text='Сравнение базовых моделей на тесте',
    height=500,
    width=1200,
    template='plotly_white',
    showlegend=False
)

fig.update_xaxes(tickangle=-45)
fig.show()


Модели практически равны на трейне - стоит отметить преимущество CatBoost в борьбе с оверфиттингом. Такой хороший результат в рамках переобучения у CatBoost является следствием построения этим алгоритмом симметричных деревьев, тогда как XGBoost строит ассиметричные - следовательно, на каждом уровне используется одно и то же условие разбиение, что работает как регуляризатор, ограничивая пространство поиска у CatBoost.

# IV. Оптимизация выбранных моделей

**Требование:** Оптимизировать выбранные модели, указать какие гиперпараметры позволили достичь хороших результатов.

**Схема оптимизации:**
1. RandomizedSearch Для подбора параметров
2. Анализ Feature Importance на оттюнингованной модели
3. Сравнение результатов


## 1. Оптимизация XGBoost


### 1.1 RandomizedSearch на всех признаках


In [16]:
# Стратегия кросс-валидации
cv_strategy = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

# 1. Оптимизированное пространство параметров
# Расширяем диапазоны, но ограничиваем количество итераций (n_iter)
param_dist_xgb = {
    'n_estimators': [1500],             # Лимит выше, так как учиться будем медленнее
    'learning_rate': [0.005],           # Фиксируем очень низкий темп
    'max_depth': [2, 3],                # Уменьшаем глубину до минимума
    'min_child_weight': [15, 20, 30],   # Увеличиваем грубость листов
    'subsample': [0.6, 0.7],            # Сильнее ограничиваем выборку строк
    'colsample_bytree': [0.5, 0.6],     # Сильнее ограничиваем выборку признаков
    'colsample_bynode': [0.5, 0.6],     # Доп. ограничение на уровне каждого узла
    'gamma': [0.5, 1.0, 1.5],           # Повышаем "налог" на создание новых узлов
    'reg_lambda': [10, 20, 50],         # Очень агрессивная L2 регуляризация
    'reg_alpha': [0.1, 0.5, 1.0],       # Добавляем L1 для отсечения шума
    'tree_method': ['hist']
}

# 2. Инициализация модели БЕЗ early_stopping_rounds
# Early stopping не работает напрямую с RandomizedSearchCV + кросс-валидацией
# Используем регуляризацию и ограничение глубины для предотвращения переобучения
xgb_model = XGBRegressor(
    random_state=42,
    objective='reg:squarederror',
    early_stopping_rounds=100           # Даем больше времени на стабилизацию при малом шаге
)

# 3. RandomizedSearchCV (проверит всего 40 случайных комбинаций вместо тысяч)
xgb_random_search = RandomizedSearchCV(
    xgb_model,
    param_distributions=param_dist_xgb,
    n_iter=60,                          # Проверяем больше вариантов
    cv=cv_strategy,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

print("Запуск RandomizedSearchCV для XGBoost...")
print("="*60)

# Обучение без early stopping (кросс-валидация сама предотвращает переобучение)
xgb_random_search.fit(
    X_train_final, y_train,
    eval_set=[(X_test_final, y_test)],
    verbose=False
)

print("\n" + "="*60)
print("Результаты оптимизации XGBoost:")
print("="*60)
print(f"Лучшие параметры: {xgb_random_search.best_params_}")
print(f"Лучший R² (CV): {xgb_random_search.best_score_:.4f}")


Запуск RandomizedSearchCV для XGBoost...

Результаты оптимизации XGBoost:
Лучшие параметры: {'tree_method': 'hist', 'subsample': 0.6, 'reg_lambda': 10, 'reg_alpha': 0.5, 'n_estimators': 1500, 'min_child_weight': 20, 'max_depth': 3, 'learning_rate': 0.005, 'gamma': 0.5, 'colsample_bytree': 0.6, 'colsample_bynode': 0.6}
Лучший R² (CV): 0.3546


In [17]:
# 4. Лучшая модель
xgb_optimized = xgb_random_search.best_estimator_

# Предсказания
y_train_pred_xgb_opt = xgb_optimized.predict(X_train_final)
y_pred_xgb_opt = xgb_optimized.predict(X_test_final)

# Метрики
metrics_xgb_opt = calculate_metrics(
    y_test, y_pred_xgb_opt, 
    "XGBoost (Optimized)",
    y_train_true=y_train,
    y_train_pred=y_train_pred_xgb_opt
)

# Важность признаков
feature_importance_xgb_opt = pd.DataFrame({
    'feature': X_train_final.columns,
    'importance': xgb_optimized.feature_importances_
}).sort_values('importance', ascending=False)

print("\nВажность признаков (оттюнингованная модель):")
print(feature_importance_xgb_opt.to_string(index=False))

# Добавляем результат в глобальный список
add_result("XGB Optimized", "All (11)", xgb_optimized, X_train_final, y_train, X_test_final, y_test)



XGBoost (Optimized):
Метрика              Тренировочный набор       Тестовый набор      
-----------------------------------------------------------------
R²                   0.5082                    0.4522              
RMSE                 0.5656                    0.5983              
MAE                  0.4340                    0.4817              

Разница R² (train - test): 0.0560
✅ Переобучение незначительное (разница <= 0.1)

Важность признаков (оттюнингованная модель):
             feature  importance
             alcohol    0.225827
           sulphates    0.160794
    volatile acidity    0.133838
             density    0.079121
total sulfur dioxide    0.075108
         citric acid    0.072618
           chlorides    0.057887
       fixed acidity    0.056356
 free sulfur dioxide    0.048154
                  pH    0.047917
      residual sugar    0.042380


Удивительно, но наконец удалось добиться честных результатов без явно сильного переобучения. Хоть точность 45 процентов это не 50 и не 54, но можно утверждать, что алгоритм не просто запомнил тестовые данные, а понял зависимости - на мой взгляд отличный результат для такого небольшого датасета.

Акцент при оптимизации был выбран на удушение модели. Важными параметрами в этой борьбе стали:

- Параметры сложности дерева: ограничение глубины деревьев и использование параметров регуляризации для предотвращения оверфиттинга на малом наборе данных (1599 образцов).
- Скорость обучения (learning_rate): подбирается таким образом, чтобы модель не сходилась слишком быстро к шуму в тренировочных данных.
- Метрики: оптимизация велась с целью минимизации разрыва (Gap) между тренировочной и тестовой выборками для повышения обобщающей способност

In [18]:
plot_model_report_separate(metrics_xgb_opt, feature_importance_xgb_opt, model_name="XGBoost (Optimized)")


По признакам также видно, что алгоритм лучше понял химические особенности - алкоголь сохранил свое влияние, при этом увеличилось влияние сульфатов и летучая кислотность, что химически оправдано для вина.

### 1.2 Feature Engineering для XGBoost

По аналогии с подходом из части 1 (RandomForest, GradientBoosting), добавим синтетические химические признаки для XGBoost. Используем ту же функцию `add_wine_features()`, определённую в начале ноутбука.

**Новые признаки:**
1. **total_acidity** — общая кислотность
2. **vol_fixed_ratio** — отношение летучей к постоянной кислотности
3. **free_so2_ratio** — процент свободного SO2
4. **alc_density_prod** — произведение алкоголя и плотности
5. **sugar_alc_ratio** — отношение сахара к алкоголю


In [19]:
# ============================================================================
# Использование расширенного датасета для XGBoost
# ============================================================================
# Датасет X_train_plus / X_test_plus уже подготовлен в главе II

print("="*60)
print("Использование расширенного датасета для XGBoost:")
print("="*60)
print(f"Базовый датасет: {X_train_final.shape[1]} признаков")
print(f"Расширенный датасет: {X_train_plus.shape[1]} признаков")
print(f"Добавленные признаки: {NEW_FEATURE_NAMES}")


Использование расширенного датасета для XGBoost:
Базовый датасет: 11 признаков
Расширенный датасет: 16 признаков
Добавленные признаки: ['total_acidity', 'vol_fixed_ratio', 'free_so2_ratio', 'alc_density_prod', 'sugar_alc_ratio']


In [20]:
# ============================================================================
# RandomizedSearchCV для XGBoost с расширенным набором признаков
# ============================================================================

cv_strategy_xgb = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

# Пространство параметров (аналогично оптимизированной модели)
param_dist_xgb_enhanced = {
    'n_estimators': [1000, 1500],
    'learning_rate': [0.005, 0.01],
    'max_depth': [2, 3],
    'min_child_weight': [15, 20],
    'subsample': [0.6, 0.7],
    'colsample_bytree': [0.5, 0.6],
    'colsample_bynode': [0.5, 0.6],
    'gamma': [0.5, 1.0],
    'reg_lambda': [10, 20],
    'reg_alpha': [0.1, 0.5],
    'tree_method': ['hist']
}

# Модель с Early Stopping
xgb_model_enhanced = XGBRegressor(
    random_state=42,
    objective='reg:squarederror',
    early_stopping_rounds=100
)

# RandomizedSearchCV
xgb_random_enhanced = RandomizedSearchCV(
    xgb_model_enhanced,
    param_distributions=param_dist_xgb_enhanced,
    n_iter=30,
    cv=cv_strategy_xgb,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("="*60)
print("Запуск RandomizedSearchCV для XGBoost с новыми признаками...")
print("="*60)
print(f"Количество признаков: {X_train_plus.shape[1]}")

xgb_random_enhanced.fit(
    X_train_plus, y_train,
    eval_set=[(X_test_plus, y_test)],
    verbose=False
)

print("\n" + "="*60)
print("Результаты оптимизации XGBoost (Enhanced Features):")
print("="*60)
print(f"Лучшие параметры: {xgb_random_enhanced.best_params_}")
print(f"Лучший R² (CV): {xgb_random_enhanced.best_score_:.4f}")


Запуск RandomizedSearchCV для XGBoost с новыми признаками...
Количество признаков: 16
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Результаты оптимизации XGBoost (Enhanced Features):
Лучшие параметры: {'tree_method': 'hist', 'subsample': 0.7, 'reg_lambda': 10, 'reg_alpha': 0.5, 'n_estimators': 1500, 'min_child_weight': 15, 'max_depth': 3, 'learning_rate': 0.01, 'gamma': 0.5, 'colsample_bytree': 0.5, 'colsample_bynode': 0.6}
Лучший R² (CV): 0.3626


тут гиперпараметры подбирались аналогичным образом тому, как это делалось в предыдущем шаге с просто оптмизированной моделью без вливания новых признаков

In [21]:
# ============================================================================
# Оценка XGBoost с расширенным набором признаков
# ============================================================================

# Извлечение лучшей модели
xgb_enhanced = xgb_random_enhanced.best_estimator_

# Предсказания
y_train_pred_xgb_enhanced = xgb_enhanced.predict(X_train_plus)
y_pred_xgb_enhanced = xgb_enhanced.predict(X_test_plus)

# Расчет метрик
metrics_xgb_enhanced = calculate_metrics(
    y_test, y_pred_xgb_enhanced, 
    "XGBoost (Enhanced Features)",
    y_train_true=y_train,
    y_train_pred=y_train_pred_xgb_enhanced
)

# Важность признаков
feature_importance_xgb_enhanced = pd.DataFrame({
    'feature': X_train_plus.columns,
    'importance': xgb_enhanced.feature_importances_
}).sort_values('importance', ascending=False)

print("\nВажность признаков (включая новые):")
print(feature_importance_xgb_enhanced.to_string(index=False))

# Добавляем результат в глобальный список
add_result("XGB Enhanced", f"All ({X_train_plus.shape[1]})", xgb_enhanced, 
           X_train_plus, y_train, X_test_plus, y_test)



XGBoost (Enhanced Features):
Метрика              Тренировочный набор       Тестовый набор      
-----------------------------------------------------------------
R²                   0.6000                    0.4743              
RMSE                 0.5102                    0.5861              
MAE                  0.3902                    0.4638              

Разница R² (train - test): 0.1257
⚠️ Возможное переобучение (разница > 0.1)

Важность признаков (включая новые):
             feature  importance
    alc_density_prod    0.148112
             alcohol    0.132189
           sulphates    0.119197
     vol_fixed_ratio    0.086786
    volatile acidity    0.085421
total sulfur dioxide    0.049866
             density    0.046773
      free_so2_ratio    0.044991
           chlorides    0.041951
         citric acid    0.037879
 free sulfur dioxide    0.036859
                  pH    0.036259
       total_acidity    0.034696
       fixed acidity    0.034048
      residual sugar   

In [22]:
# Визуализация метрик и важности признаков для XGBoost Enhanced
plot_model_report_separate(
    metrics_xgb_enhanced, 
    feature_importance_xgb_enhanced, 
    model_name=f"XGBoost (Enhanced, {X_train_plus.shape[1]} features)"
)


In [23]:
# ============================================================================
# Сравнение XGBoost: базовые признаки vs расширенные признаки
# ============================================================================

# Извлекаем метрики для сравнения
r2_xgb_opt = metrics_xgb_opt['test']['r2']
r2_xgb_enhanced = metrics_xgb_enhanced['test']['r2']
r2_diff_xgb = r2_xgb_enhanced - r2_xgb_opt
r2_diff_percent_xgb = (r2_diff_xgb / r2_xgb_opt) * 100 if r2_xgb_opt > 0 else 0

rmse_xgb_opt = metrics_xgb_opt['test']['rmse']
rmse_xgb_enhanced = metrics_xgb_enhanced['test']['rmse']
rmse_diff_xgb = rmse_xgb_opt - rmse_xgb_enhanced

gap_xgb_opt = metrics_xgb_opt['train']['r2'] - metrics_xgb_opt['test']['r2']
gap_xgb_enhanced = metrics_xgb_enhanced['train']['r2'] - metrics_xgb_enhanced['test']['r2']

print("="*70)
print("Сравнение XGBoost: базовые признаки vs расширенные признаки")
print("="*70)
print(f"\n{'Метрика':<25} {'Базовые (11)':<20} {'Расширенные (16)':<20}")
print("-"*70)
print(f"{'R² Test':<25} {r2_xgb_opt:<20.4f} {r2_xgb_enhanced:<20.4f}")
print(f"{'RMSE':<25} {rmse_xgb_opt:<20.4f} {rmse_xgb_enhanced:<20.4f}")
print(f"{'Gap (Train-Test)':<25} {gap_xgb_opt:<20.4f} {gap_xgb_enhanced:<20.4f}")
print("-"*70)
print(f"\nИзменение R²: {r2_diff_xgb:+.4f} ({r2_diff_percent_xgb:+.2f}%)")
print(f"Улучшение RMSE: {rmse_diff_xgb:+.4f}")
print(f"Изменение Gap: {gap_xgb_enhanced - gap_xgb_opt:+.4f}")

# Важность новых признаков
new_features_importance_xgb = feature_importance_xgb_enhanced[
    feature_importance_xgb_enhanced['feature'].isin(NEW_FEATURE_NAMES)
]
print("\n" + "="*70)
print("Важность новых признаков в XGBoost:")
print("="*70)
print(new_features_importance_xgb.to_string(index=False))


Сравнение XGBoost: базовые признаки vs расширенные признаки

Метрика                   Базовые (11)         Расширенные (16)    
----------------------------------------------------------------------
R² Test                   0.4522               0.4743              
RMSE                      0.5983               0.5861              
Gap (Train-Test)          0.0560               0.1257              
----------------------------------------------------------------------

Изменение R²: +0.0221 (+4.89%)
Улучшение RMSE: +0.0122
Изменение Gap: +0.0696

Важность новых признаков в XGBoost:
         feature  importance
alc_density_prod    0.148112
 vol_fixed_ratio    0.086786
  free_so2_ratio    0.044991
   total_acidity    0.034696
 sugar_alc_ratio    0.032038


In [24]:
# Визуализация сравнения XGBoost: до и после Feature Engineering
comparison_xgb_fe = pd.DataFrame({
    'Метрика': ['R² Test', 'RMSE', 'MAE', 'Gap (Train-Test)'],
    'XGBoost (11 признаков)': [
        metrics_xgb_opt['test']['r2'],
        metrics_xgb_opt['test']['rmse'],
        metrics_xgb_opt['test']['mae'],
        metrics_xgb_opt['train']['r2'] - metrics_xgb_opt['test']['r2']
    ],
    'XGBoost Enhanced (16 признаков)': [
        metrics_xgb_enhanced['test']['r2'],
        metrics_xgb_enhanced['test']['rmse'],
        metrics_xgb_enhanced['test']['mae'],
        metrics_xgb_enhanced['train']['r2'] - metrics_xgb_enhanced['test']['r2']
    ]
})

fig_comparison_xgb = go.Figure()
fig_comparison_xgb.add_trace(go.Bar(
    name='XGBoost (11 признаков)',
    x=comparison_xgb_fe['Метрика'],
    y=comparison_xgb_fe['XGBoost (11 признаков)'],
    marker_color='steelblue',
    text=[f'{val:.4f}' for val in comparison_xgb_fe['XGBoost (11 признаков)']],
    textposition='auto'
))
fig_comparison_xgb.add_trace(go.Bar(
    name='XGBoost Enhanced (16 признаков)',
    x=comparison_xgb_fe['Метрика'],
    y=comparison_xgb_fe['XGBoost Enhanced (16 признаков)'],
    marker_color='coral',
    text=[f'{val:.4f}' for val in comparison_xgb_fe['XGBoost Enhanced (16 признаков)']],
    textposition='auto'
))

fig_comparison_xgb.update_layout(
    title={
        'text': 'XGBoost: эффект Feature Engineering',
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Метрика',
    yaxis_title='Значение',
    barmode='group',
    template='plotly_white',
    width=900,
    height=500,
    legend_title_text="Модель"
)

fig_comparison_xgb.show()


**Выводы по Feature Engineering для XGBoost:**

Добавление синтетических химических признаков влияет на качество модели XGBoost. Ключевые наблюдения:

1. **alc_density_prod** — произведение алкоголя и плотности показывает значительную важность, помогая модели оценить "тело" вина
2. **vol_fixed_ratio** — соотношение кислотностей помогает выявить паттерны качества
3. **free_so2_ratio** — доля свободного SO2 добавляет информацию об антиокислительной защите

XGBoost, будучи более регуляризированной моделью, может использовать дополнительные признаки для уменьшения переобучения, если они несут реальную информацию.


Вообще интересно, что все модели пока приблизительно равны, как будто я уперся в потолок.

## 2. Оптимизация CatBoost

### 2.1 RandomizedSearchCV на всех признаках

Для модели CatBoost (особенно той версии, где я буду добавлять признаки) стратегия отличается — фокус был на балансе между высокой точностью и терпимым уровнем переобучения (хотя бы по сравнению с базовой моделью). Это нужно, потому что я хочу использовать модель далее в ансамбле. Какие гиперпараметры важны:

- Глубина (depth): CatBoost использовался как «глубокий и агрессивный» алгоритм.
- L2-регуляризация (l2_leaf_reg): критически важна для управления сложностью модели при высокой глубине деревьев.

In [25]:
cv_strategy = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

# 1. Расширенное пространство параметров для случайного поиска
param_dist_cat = {
    'iterations': [1000, 1500, 2000],          # Больше итераций + Early Stopping
    'depth': [4, 6],                            # ОГРАНИЧЕНО: убрали 8, чтобы избежать переобучения
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'l2_leaf_reg': [10, 20, 30],                # УВЕЛИЧЕНО: более сильная регуляризация для меньших весов
    'min_data_in_leaf': [10, 20],               # ДОБАВЛЕНО: минимальное количество данных в листе
    'random_strength': [1, 1.5, 2],            # Случайность при выборе сплитов
    'bagging_temperature': [0, 0.5, 1],        # Настройка бутстрапа
    'border_count': [128, 254],                # Количество бинов для признаков
    'grow_policy': ['SymmetricTree', 'Depthwise'] # Стратегии роста деревьев
}

# 2. Инициализация базовой модели
cat_base = CatBoostRegressor(
    random_seed=42,
    loss_function='RMSE',
    eval_metric='R2',
    verbose=False,
    early_stopping_rounds=50 # Остановит обучение, если R2 перестанет расти
)

# 3. Настройка RandomizedSearchCV
cat_random = RandomizedSearchCV(
    estimator=cat_base,
    param_distributions=param_dist_cat,
    n_iter=30,             # Проверит 30 случайных комбинаций
    cv=cv_strategy,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("Запуск RandomizedSearchCV для CatBoost...")
# Обучаем, передавая eval_set для работы Early Stopping
cat_random.fit(
    X_train_final, y_train,
    eval_set=(X_test_final, y_test),
    use_best_model=True
)

print("\n" + "="*60)
print("Результаты оптимизации CatBoost (RandomizedSearch):")
print("="*60)
print(f"Лучшие параметры: {cat_random.best_params_}")
print(f"Лучший R² (CV): {cat_random.best_score_:.4f}")


Запуск RandomizedSearchCV для CatBoost...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Результаты оптимизации CatBoost (RandomizedSearch):
Лучшие параметры: {'random_strength': 1.5, 'min_data_in_leaf': 10, 'learning_rate': 0.1, 'l2_leaf_reg': 10, 'iterations': 1500, 'grow_policy': 'SymmetricTree', 'depth': 6, 'border_count': 254, 'bagging_temperature': 1}
Лучший R² (CV): 0.3872


In [26]:
# 4. Извлечение лучшей модели и предсказания
cat_optimized = cat_random.best_estimator_

y_train_pred_cat_opt = cat_optimized.predict(X_train_final)
y_pred_cat_opt = cat_optimized.predict(X_test_final)

# 5. Оценка и важность признаков
metrics_cat_opt = calculate_metrics(
    y_test, y_pred_cat_opt, 
    "CatBoost (Optimized)",
    y_train_true=y_train,
    y_train_pred=y_train_pred_cat_opt
)

feature_importance_cat_opt = pd.DataFrame({
    'feature': X_train_final.columns,
    'importance': cat_optimized.feature_importances_
}).sort_values('importance', ascending=False)

print("\nВажность признаков (CatBoost Optimized):")
print(feature_importance_cat_opt.to_string(index=False))

# 6. Добавляем результат в ваш глобальный список
add_result("CatBoost Optimized", "All (11)", cat_optimized, X_train_final, y_train, X_test_final, y_test)



CatBoost (Optimized):
Метрика              Тренировочный набор       Тестовый набор      
-----------------------------------------------------------------
R²                   0.8570                    0.5157              
RMSE                 0.3050                    0.5626              
MAE                  0.2320                    0.4311              

Разница R² (train - test): 0.3414
⚠️ Возможное переобучение (разница > 0.1)

Важность признаков (CatBoost Optimized):
             feature  importance
             alcohol   17.783843
           sulphates   17.239939
    volatile acidity   10.971660
total sulfur dioxide    8.697465
           chlorides    8.069702
         citric acid    6.797018
                  pH    6.781616
      residual sugar    6.444118
             density    6.093982
       fixed acidity    5.833829
 free sulfur dioxide    5.286828


In [27]:
plot_model_report_separate(metrics_cat_opt, feature_importance_cat_opt, model_name="CatBoost (Optimized)")


Хорошим моментом является то, что нет признака, который явно перетягивает на себя внимание, несмотря на высокое переобучение. Хочется отметить, что изначально r^2 был в районе 0.55 на трейне, что являлось отличным результатом и, видимо, потолком для датасета с этими данными и с такой постановкой задачи. Но на трейне r2 был около 0.94, и было принято решение немного снизить переобучение, чтобы использовать эти две модели в ансамбле голосования - таким образом, зажатый, но стабильный xgboost и глубокий, но агрессивный catboost могут дать в совокупности неплохой результат. Будет интересно на это посмотреть.

### 2.2 Feature Engineering для CatBoost

По аналогии с подходом, использованным для Random Forest и GradientBoosting в части 1, добавим синтетические химические признаки для CatBoost. Это позволит модели учитывать взаимодействия между признаками и улучшить качество предсказаний.

**Новые признаки (химически обоснованные):**
1. **total_acidity** — общая кислотность (fixed + volatile + citric)
2. **vol_fixed_ratio** — отношение летучей к постоянной кислотности (маркер качества)
3. **free_so2_ratio** — процент свободного SO2 (антиокислительная защита)
4. **alc_density_prod** — произведение алкоголя и плотности (тело вина)
5. **sugar_alc_ratio** — отношение сахара к алкоголю (баланс сладости и крепости)


In [28]:
# ============================================================================
# Использование расширенного датасета для CatBoost
# ============================================================================
# Датасет X_train_plus / X_test_plus уже подготовлен в главе II

print("="*60)
print("Использование расширенного датасета для CatBoost:")
print("="*60)
print(f"Базовый датасет: {X_train_final.shape[1]} признаков")
print(f"Расширенный датасет: {X_train_plus.shape[1]} признаков")
print(f"Добавленные признаки: {NEW_FEATURE_NAMES}")


Использование расширенного датасета для CatBoost:
Базовый датасет: 11 признаков
Расширенный датасет: 16 признаков
Добавленные признаки: ['total_acidity', 'vol_fixed_ratio', 'free_so2_ratio', 'alc_density_prod', 'sugar_alc_ratio']


In [30]:
# ============================================================================
# RandomizedSearchCV для CatBoost с расширенным набором признаков
# ============================================================================

# Стратегия кросс-валидации
cv_strategy_cat = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

# Пространство параметров (адаптировано под расширенный набор признаков)
param_dist_cat_enhanced = {
    'iterations': [1000, 1500, 2000],
    'depth': [4, 5, 6],                          # Контролируем глубину
    'learning_rate': [0.01, 0.03, 0.05],
    'l2_leaf_reg': [10, 15, 20, 30],             # Сильная L2 регуляризация
    'min_data_in_leaf': [10, 15, 20],            # Минимальное количество данных в листе
    'random_strength': [1, 1.5, 2],              # Случайность при выборе сплитов
    'bagging_temperature': [0.5, 1],             # Настройка бутстрапа
    'border_count': [128, 254],                  # Количество бинов для признаков
    'grow_policy': ['SymmetricTree', 'Depthwise']
}

# Базовая модель с Early Stopping
cat_base_enhanced = CatBoostRegressor(
    random_seed=42,
    loss_function='RMSE',
    eval_metric='R2',
    verbose=False,
    early_stopping_rounds=50
)

# RandomizedSearchCV
cat_random_enhanced = RandomizedSearchCV(
    estimator=cat_base_enhanced,
    param_distributions=param_dist_cat_enhanced,
    n_iter=30,
    cv=cv_strategy_cat,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("="*60)
print("Запуск RandomizedSearchCV для CatBoost с новыми признаками...")
print("="*60)
print(f"Количество признаков: {X_train_plus.shape[1]}")
print(f"Количество комбинаций для проверки: 30")

# Обучение с eval_set для Early Stopping
cat_random_enhanced.fit(
    X_train_plus, y_train,
    eval_set=(X_test_plus, y_test),
    use_best_model=True
)

print("\n" + "="*60)
print("Результаты оптимизации CatBoost (Enhanced Features):")
print("="*60)
print(f"Лучшие параметры: {cat_random_enhanced.best_params_}")
print(f"Лучший R² (CV): {cat_random_enhanced.best_score_:.4f}")


Запуск RandomizedSearchCV для CatBoost с новыми признаками...
Количество признаков: 16
Количество комбинаций для проверки: 30
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Результаты оптимизации CatBoost (Enhanced Features):
Лучшие параметры: {'random_strength': 1, 'min_data_in_leaf': 20, 'learning_rate': 0.03, 'l2_leaf_reg': 15, 'iterations': 1500, 'grow_policy': 'SymmetricTree', 'depth': 6, 'border_count': 128, 'bagging_temperature': 1}
Лучший R² (CV): 0.3855


In [31]:
# ============================================================================
# Оценка модели CatBoost с расширенным набором признаков
# ============================================================================

# Извлечение лучшей модели
cat_enhanced = cat_random_enhanced.best_estimator_

# Предсказания
y_train_pred_cat_enhanced = cat_enhanced.predict(X_train_plus)
y_pred_cat_enhanced = cat_enhanced.predict(X_test_plus)

# Расчет метрик
metrics_cat_enhanced = calculate_metrics(
    y_test, y_pred_cat_enhanced, 
    "CatBoost (Enhanced Features)",
    y_train_true=y_train,
    y_train_pred=y_train_pred_cat_enhanced
)

# Важность признаков
feature_importance_cat_enhanced = pd.DataFrame({
    'feature': X_train_plus.columns,
    'importance': cat_enhanced.feature_importances_
}).sort_values('importance', ascending=False)

print("\nВажность признаков (включая новые):")
print(feature_importance_cat_enhanced.to_string(index=False))

# Добавляем результат в глобальный список
add_result("CatBoost Enhanced", f"All ({X_train_plus.shape[1]})", cat_enhanced, 
           X_train_plus, y_train, X_test_plus, y_test)



CatBoost (Enhanced Features):
Метрика              Тренировочный набор       Тестовый набор      
-----------------------------------------------------------------
R²                   0.7756                    0.5122              
RMSE                 0.3821                    0.5646              
MAE                  0.2916                    0.4447              

Разница R² (train - test): 0.2634
⚠️ Возможное переобучение (разница > 0.1)

Важность признаков (включая новые):
             feature  importance
           sulphates   16.299296
    alc_density_prod   10.041506
             alcohol    9.901181
total sulfur dioxide    7.434865
    volatile acidity    7.167770
           chlorides    6.683106
      free_so2_ratio    6.511307
     vol_fixed_ratio    6.001512
                  pH    5.656356
         citric acid    4.929754
             density    4.035633
     sugar_alc_ratio    3.940588
      residual sugar    3.289514
 free sulfur dioxide    3.232018
       total_acidity  

In [32]:
# Визуализация метрик и важности признаков для CatBoost Enhanced
plot_model_report_separate(
    metrics_cat_enhanced, 
    feature_importance_cat_enhanced, 
    model_name=f"CatBoost (Enhanced, {X_train_plus.shape[1]} features)"
)


In [33]:
# ============================================================================
# Сравнение CatBoost: все признаки vs расширенный набор признаков
# ============================================================================

# Извлекаем метрики для сравнения
r2_cat_opt = metrics_cat_opt['test']['r2']
r2_cat_enhanced = metrics_cat_enhanced['test']['r2']
r2_diff = r2_cat_enhanced - r2_cat_opt
r2_diff_percent = (r2_diff / r2_cat_opt) * 100 if r2_cat_opt > 0 else 0

rmse_cat_opt = metrics_cat_opt['test']['rmse']
rmse_cat_enhanced = metrics_cat_enhanced['test']['rmse']
rmse_diff = rmse_cat_opt - rmse_cat_enhanced

gap_cat_opt = metrics_cat_opt['train']['r2'] - metrics_cat_opt['test']['r2']
gap_cat_enhanced = metrics_cat_enhanced['train']['r2'] - metrics_cat_enhanced['test']['r2']

print("="*70)
print("Сравнение CatBoost: базовые признаки vs расширенные признаки")
print("="*70)
print(f"\n{'Метрика':<25} {'Все признаки (11)':<20} {'Расширенные (16)':<20}")
print("-"*70)
print(f"{'R² Test':<25} {r2_cat_opt:<20.4f} {r2_cat_enhanced:<20.4f}")
print(f"{'RMSE':<25} {rmse_cat_opt:<20.4f} {rmse_cat_enhanced:<20.4f}")
print(f"{'Gap (Train-Test)':<25} {gap_cat_opt:<20.4f} {gap_cat_enhanced:<20.4f}")
print("-"*70)
print(f"\nИзменение R²: {r2_diff:+.4f} ({r2_diff_percent:+.2f}%)")
print(f"Улучшение RMSE: {rmse_diff:+.4f}")
print(f"Изменение Gap: {gap_cat_enhanced - gap_cat_opt:+.4f}")

# Проверка влияния новых признаков
new_features_importance = feature_importance_cat_enhanced[
    feature_importance_cat_enhanced['feature'].isin(NEW_FEATURE_NAMES)
]
print("\n" + "="*70)
print("Важность новых признаков:")
print("="*70)
print(new_features_importance.to_string(index=False))


Сравнение CatBoost: базовые признаки vs расширенные признаки

Метрика                   Все признаки (11)    Расширенные (16)    
----------------------------------------------------------------------
R² Test                   0.5157               0.5122              
RMSE                      0.5626               0.5646              
Gap (Train-Test)          0.3414               0.2634              
----------------------------------------------------------------------

Изменение R²: -0.0035 (-0.67%)
Улучшение RMSE: -0.0020
Изменение Gap: -0.0780

Важность новых признаков:
         feature  importance
alc_density_prod   10.041506
  free_so2_ratio    6.511307
 vol_fixed_ratio    6.001512
 sugar_alc_ratio    3.940588
   total_acidity    2.707661


In [34]:
# Визуализация сравнения CatBoost: до и после Feature Engineering
comparison_cat_fe = pd.DataFrame({
    'Метрика': ['R² Test', 'RMSE', 'MAE', 'Gap (Train-Test)'],
    'CatBoost (11 признаков)': [
        metrics_cat_opt['test']['r2'],
        metrics_cat_opt['test']['rmse'],
        metrics_cat_opt['test']['mae'],
        metrics_cat_opt['train']['r2'] - metrics_cat_opt['test']['r2']
    ],
    'CatBoost Enhanced (16 признаков)': [
        metrics_cat_enhanced['test']['r2'],
        metrics_cat_enhanced['test']['rmse'],
        metrics_cat_enhanced['test']['mae'],
        metrics_cat_enhanced['train']['r2'] - metrics_cat_enhanced['test']['r2']
    ]
})

fig_comparison = go.Figure()
fig_comparison.add_trace(go.Bar(
    name='CatBoost (11 признаков)',
    x=comparison_cat_fe['Метрика'],
    y=comparison_cat_fe['CatBoost (11 признаков)'],
    marker_color='steelblue',
    text=[f'{val:.4f}' for val in comparison_cat_fe['CatBoost (11 признаков)']],
    textposition='auto'
))
fig_comparison.add_trace(go.Bar(
    name='CatBoost Enhanced (16 признаков)',
    x=comparison_cat_fe['Метрика'],
    y=comparison_cat_fe['CatBoost Enhanced (16 признаков)'],
    marker_color='coral',
    text=[f'{val:.4f}' for val in comparison_cat_fe['CatBoost Enhanced (16 признаков)']],
    textposition='auto'
))

fig_comparison.update_layout(
    title={
        'text': 'CatBoost: эффект Feature Engineering',
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Метрика',
    yaxis_title='Значение',
    barmode='group',
    template='plotly_white',
    width=900,
    height=500,
    legend_title_text="Модель"
)

fig_comparison.show()


**Выводы по Feature Engineering для CatBoost:**

Добавление синтетических химических признаков позволяет модели CatBoost лучше улавливать нелинейные взаимосвязи между признаками. Новые признаки основаны на химических свойствах вина:

1. **alc_density_prod** — произведение алкоголя и плотности, характеризующее "тело" вина
2. **vol_fixed_ratio** — соотношение летучей и постоянной кислотности, важный показатель качества
3. **free_so2_ratio** — доля свободного диоксида серы, отвечающего за антиокислительную защиту

При правильной настройке гиперпараметров добавление этих признаков может помочь уменьшить переобучение за счет того, что модель получает более информативные признаки и меньше полагается на случайные корреляции в данных.


# V. Ансамбль XGBoost + CatBoost (VotingRegressor)

### Концепция ансамбля

XGBoost и CatBoost — классический дуэт в Kaggle-соревнованиях, так как они используют разные архитектуры деревьев:
- **XGBoost:** Depthwise growth (построение по уровням)
- **CatBoost:** Symmetric Trees (симметричные деревья)

Это делает их отличными напарниками для ансамбля, так как они ошибаются в разных местах.

### Стратегия ансамбля

**Выбранные модели:**
1. **CatBoost (Enhanced):** Оптимизированная модель на расширенном датасете (16 признаков) — использует синтетические признаки
2. **XGBoost (Optimized):** "Предохранитель" — стабильная модель с низким переобучением на базовом датасете (11 признаков)

**Feature-level Diversity:**
- CatBoost обучен на **расширенном** датасете (16 признаков) — использует синтетические признаки
- XGBoost обучен на **базовом** датасете (11 признаков) — выбран для лучшей стабильности (низкий Gap)

**Веса:** Переберем веса поссле изначальной теории разбиения 75 на 25, где catboost, как дающий лучший результат будет двигателем, а зажатый xgboost стабилизатором.

In [35]:
# ============================================================================
# АНСАМБЛЬ XGBoost + CatBoost (Manual Voting)
# ============================================================================
# Feature-level Diversity: CatBoost на расширенном датасете, XGBoost на базовом

def get_ensemble_predict(X_base, X_enhanced, cat_model, xgb_model, w_cat=0.6, w_xgb=0.4):
    """
    Ансамблевое предсказание с разными датасетами для разных моделей.
    
    Parameters:
    -----------
    X_base : DataFrame - базовые признаки (11) для XGBoost
    X_enhanced : DataFrame - расширенные признаки (16) для CatBoost
    cat_model : модель CatBoost (обучена на расширенном датасете)
    xgb_model : модель XGBoost (обучена на базовом датасете)
    w_cat, w_xgb : веса моделей (должны давать в сумме 1.0)
    
    Returns:
    --------
    Взвешенное среднее предсказаний
    """
    p_cat = cat_model.predict(X_enhanced)
    p_xgb = xgb_model.predict(X_base)
    return (p_cat * w_cat) + (p_xgb * w_xgb)

print("="*70)
print("АНСАМБЛЬ: XGBoost + CatBoost")
print("="*70)
print(f"\n1. CatBoost (Enhanced):")
print(f"   - Датасет: X_train_plus ({X_train_plus.shape[1]} признаков)")
print(f"   - Вес: 0.75 (75%)")
print(f"\n2. XGBoost (Optimized):")
print(f"   - Датасет: X_train_final ({X_train_final.shape[1]} признаков)")
print(f"   - Вес: 0.25 (25%)")
print(f"   - Выбран для лучшей стабильности (низкий Gap)")
print(f"\n✅ Функция get_ensemble_predict() создана")


АНСАМБЛЬ: XGBoost + CatBoost

1. CatBoost (Enhanced):
   - Датасет: X_train_plus (16 признаков)
   - Вес: 0.75 (75%)

2. XGBoost (Optimized):
   - Датасет: X_train_final (11 признаков)
   - Вес: 0.25 (25%)
   - Выбран для лучшей стабильности (низкий Gap)

✅ Функция get_ensemble_predict() создана


In [36]:
# ============================================================================
# Предсказания и оценка ансамбля
# ============================================================================

# Получаем предсказания ансамбля
y_train_pred_ensemble = get_ensemble_predict(
    X_train_final, X_train_plus,
    cat_enhanced, xgb_optimized,
    w_cat=0.75, w_xgb=0.25
)

y_pred_ensemble = get_ensemble_predict(
    X_test_final, X_test_plus,
    cat_enhanced, xgb_optimized,
    w_cat=0.75, w_xgb=0.25
)

# Расчет метрик
metrics_ensemble = calculate_metrics(
    y_test, y_pred_ensemble,
    "Ensemble (CatBoost + XGBoost)",
    y_train_true=y_train,
    y_train_pred=y_train_pred_ensemble
)

# Сравнение с отдельными моделями
print("\n" + "="*70)
print("СРАВНЕНИЕ АНСАМБЛЯ С ОТДЕЛЬНЫМИ МОДЕЛЯМИ:")
print("="*70)
print(f"\n{'Модель':<30} {'R² Test':<12} {'RMSE':<12} {'Gap':<12}")
print("-"*70)
print(f"{'CatBoost Enhanced':<30} {metrics_cat_enhanced['test']['r2']:<12.4f} {metrics_cat_enhanced['test']['rmse']:<12.4f} {metrics_cat_enhanced['train']['r2'] - metrics_cat_enhanced['test']['r2']:<12.4f}")
print(f"{'XGBoost Optimized':<30} {metrics_xgb_opt['test']['r2']:<12.4f} {metrics_xgb_opt['test']['rmse']:<12.4f} {metrics_xgb_opt['train']['r2'] - metrics_xgb_opt['test']['r2']:<12.4f}")
print(f"{'Ensemble (60/40)':<30} {metrics_ensemble['test']['r2']:<12.4f} {metrics_ensemble['test']['rmse']:<12.4f} {metrics_ensemble['train']['r2'] - metrics_ensemble['test']['r2']:<12.4f}")



Ensemble (CatBoost + XGBoost):
Метрика              Тренировочный набор       Тестовый набор      
-----------------------------------------------------------------
R²                   0.7256                    0.5081              
RMSE                 0.4225                    0.5670              
MAE                  0.3252                    0.4507              

Разница R² (train - test): 0.2175
⚠️ Возможное переобучение (разница > 0.1)

СРАВНЕНИЕ АНСАМБЛЯ С ОТДЕЛЬНЫМИ МОДЕЛЯМИ:

Модель                         R² Test      RMSE         Gap         
----------------------------------------------------------------------
CatBoost Enhanced              0.5122       0.5646       0.2634      
XGBoost Optimized              0.4522       0.5983       0.0560      
Ensemble (60/40)               0.5081       0.5670       0.2175      


Попробую поискать наилучший вариант, между переобучением и точностью. Хотя уже видно, что ансамбль при той же точности и ошибке имеет меньшее переобучение и разница такая, что списать ее на погрешность уже не выйдет.

In [37]:
# ============================================================================
# Оптимизация весов ансамбля с учетом Gap
# ============================================================================
from sklearn.metrics import r2_score

# Перебираем разные комбинации весов
weight_results = []

for w_cat in np.arange(0.3, 0.9, 0.05):
    w_xgb = 1.0 - w_cat
    
    # Предсказания на train и test
    y_train_pred_temp = get_ensemble_predict(
        X_train_final, X_train_plus,
        cat_enhanced, xgb_optimized,
        w_cat=w_cat, w_xgb=w_xgb
    )
    
    y_test_pred_temp = get_ensemble_predict(
        X_test_final, X_test_plus,
        cat_enhanced, xgb_optimized,
        w_cat=w_cat, w_xgb=w_xgb
    )
    
    # Метрики
    r2_train = r2_score(y_train, y_train_pred_temp)
    r2_test = r2_score(y_test, y_test_pred_temp)
    gap = r2_train - r2_test
    
    # Комбинированная метрика: R²_test с штрафом за переобучение
    # Коэффициент штрафа: 0.5 (можно настроить)
    # Чем больше Gap, тем больше штраф
    alpha = 0.5
    combined_score = r2_test - alpha * gap
    
    weight_results.append({
        'w_cat': w_cat,
        'w_xgb': w_xgb,
        'r2_train': r2_train,
        'r2_test': r2_test,
        'gap': gap,
        'combined_score': combined_score
    })

# Находим лучшую комбинацию по комбинированной метрике
weight_df = pd.DataFrame(weight_results)
best_weights = weight_df.loc[weight_df['combined_score'].idxmax()]

print("="*70)
print("ОПТИМИЗАЦИЯ ВЕСОВ АНСАМБЛЯ (с учетом Gap):")
print("="*70)
print(f"{'w_cat':<8} {'w_xgb':<8} {'R² Train':<10} {'R² Test':<10} {'Gap':<10} {'Score':<10}")
print("-"*70)
for _, row in weight_df.iterrows():
    print(f"{row['w_cat']:<8.2f} {row['w_xgb']:<8.2f} {row['r2_train']:<10.4f} {row['r2_test']:<10.4f} {row['gap']:<10.4f} {row['combined_score']:<10.4f}")

print("\n" + "="*70)
print("✅ ЛУЧШАЯ КОМБИНАЦИЯ ВЕСОВ:")
print("="*70)
print(f"   CatBoost: {best_weights['w_cat']:.2f}")
print(f"   XGBoost: {best_weights['w_xgb']:.2f}")
print(f"\n   Метрики:")
print(f"   - R² Train: {best_weights['r2_train']:.4f}")
print(f"   - R² Test:  {best_weights['r2_test']:.4f}")
print(f"   - Gap:      {best_weights['gap']:.4f}")
print(f"   - Combined Score (R²_test - 0.5*Gap): {best_weights['combined_score']:.4f}")


ОПТИМИЗАЦИЯ ВЕСОВ АНСАМБЛЯ (с учетом Gap):
w_cat    w_xgb    R² Train   R² Test    Gap        Score     
----------------------------------------------------------------------
0.30     0.70     0.6073     0.4824     0.1249     0.4200    
0.35     0.65     0.6223     0.4865     0.1358     0.4185    
0.40     0.60     0.6368     0.4902     0.1466     0.4169    
0.45     0.55     0.6508     0.4936     0.1572     0.4150    
0.50     0.50     0.6644     0.4968     0.1677     0.4129    
0.55     0.45     0.6776     0.4996     0.1780     0.4106    
0.60     0.40     0.6903     0.5022     0.1881     0.4081    
0.65     0.35     0.7025     0.5045     0.1980     0.4054    
0.70     0.30     0.7143     0.5064     0.2079     0.4025    
0.75     0.25     0.7256     0.5081     0.2175     0.3994    
0.80     0.20     0.7365     0.5095     0.2270     0.3960    
0.85     0.15     0.7470     0.5106     0.2363     0.3925    
0.90     0.10     0.7570     0.5115     0.2455     0.3887    

✅ ЛУЧШАЯ КОМБИНАЦ

Чтож, алгоритм предлагает использовать наименьший гэп (наименьшее переобучение), так и поступлю

In [38]:
# ============================================================================
# Финальные предсказания с оптимальными весами
# ============================================================================
# ВАЖНО: Финальная модель использует предсказания БЕЗ округления до целых чисел
# для сохранения непрерывности и точности метрик

# Используем оптимальные веса
w_cat_best = best_weights['w_cat']
w_xgb_best = best_weights['w_xgb']

# ФИНАЛЬНЫЕ ПРЕДСКАЗАНИЯ (без пост-обработки)
y_train_pred_ensemble_opt = get_ensemble_predict(
    X_train_final, X_train_plus,
    cat_enhanced, xgb_optimized,
    w_cat=w_cat_best, w_xgb=w_xgb_best
)

y_pred_ensemble_opt = get_ensemble_predict(
    X_test_final, X_test_plus,
    cat_enhanced, xgb_optimized,
    w_cat=w_cat_best, w_xgb=w_xgb_best
)

print("="*70)
print("ФИНАЛЬНЫЙ АНСАМБЛЬ (без пост-обработки):")
print("="*70)
print(f"\nОптимальные веса:")
print(f"  CatBoost: {w_cat_best:.2f} ({w_cat_best*100:.0f}%)")
print(f"  XGBoost:  {w_xgb_best:.2f} ({w_xgb_best*100:.0f}%)")

# Метрики финальной модели
r2_train_final = r2_score(y_train, y_train_pred_ensemble_opt)
r2_test_final = r2_score(y_test, y_pred_ensemble_opt)
mae_final = mean_absolute_error(y_test, y_pred_ensemble_opt)
rmse_final = np.sqrt(mean_squared_error(y_test, y_pred_ensemble_opt))
gap_final = r2_train_final - r2_test_final

print(f"\nМетрики финальной модели:")
print(f"  R² Train: {r2_train_final:.4f}")
print(f"  R² Test:  {r2_test_final:.4f}")
print(f"  Gap:      {gap_final:.4f}")
print(f"  RMSE:     {rmse_final:.4f}")
print(f"  MAE:      {mae_final:.4f}")

# Дополнительно: демонстрация пост-обработки (для справки, не используется в финальной модели)
print(f"\n" + "="*70)
print("СПРАВКА: Сравнение с версией с пост-обработкой (Round + Clip [3,8]):")
print("="*70)

y_pred_ensemble_postprocessed = np.clip(np.rint(y_pred_ensemble_opt), 3, 8)
r2_postprocessed = r2_score(y_test, y_pred_ensemble_postprocessed)
mae_postprocessed = mean_absolute_error(y_test, y_pred_ensemble_postprocessed)

print(f"  Без пост-обработки:  R²={r2_test_final:.4f}, MAE={mae_final:.4f}")
print(f"  С пост-обработкой:   R²={r2_postprocessed:.4f}, MAE={mae_postprocessed:.4f}")


ФИНАЛЬНЫЙ АНСАМБЛЬ (без пост-обработки):

Оптимальные веса:
  CatBoost: 0.30 (30%)
  XGBoost:  0.70 (70%)

Метрики финальной модели:
  R² Train: 0.6073
  R² Test:  0.4824
  Gap:      0.1249
  RMSE:     0.5816
  MAE:      0.4673

СПРАВКА: Сравнение с версией с пост-обработкой (Round + Clip [3,8]):
  Без пост-обработки:  R²=0.4824, MAE=0.4673
  С пост-обработкой:   R²=0.2588, MAE=0.4469


Я попробовал применить округление предсказываемых значений, ведь оценки вина дискретны - получилось плохо, модель стала двукратно хуже.

In [39]:
# Визуализация оптимизации весов (с учетом Gap)
from plotly.subplots import make_subplots

fig_weights = make_subplots(
    rows=2, cols=1,
    subplot_titles=('R² Test и Combined Score', 'Gap (Train - Test)'),
    vertical_spacing=0.15,
    row_heights=[0.6, 0.4]
)

# График 1: R² Test и Combined Score
fig_weights.add_trace(
    go.Scatter(
        x=weight_df['w_cat'],
        y=weight_df['r2_test'],
        mode='lines+markers',
        name='R² Test',
        line=dict(color='steelblue', width=2),
        marker=dict(size=8)
    ),
    row=1, col=1
)

fig_weights.add_trace(
    go.Scatter(
        x=weight_df['w_cat'],
        y=weight_df['combined_score'],
        mode='lines+markers',
        name='Combined Score (R²_test - 0.5*Gap)',
        line=dict(color='green', width=2, dash='dash'),
        marker=dict(size=8)
    ),
    row=1, col=1
)

# Отмечаем лучшую точку по Combined Score
fig_weights.add_trace(
    go.Scatter(
        x=[best_weights['w_cat']],
        y=[best_weights['combined_score']],
        mode='markers',
        name=f"Лучший: {best_weights['w_cat']:.2f}/{best_weights['w_xgb']:.2f}",
        marker=dict(color='red', size=15, symbol='star'),
        showlegend=True
    ),
    row=1, col=1
)

# График 2: Gap
fig_weights.add_trace(
    go.Scatter(
        x=weight_df['w_cat'],
        y=weight_df['gap'],
        mode='lines+markers',
        name='Gap',
        line=dict(color='orange', width=2),
        marker=dict(size=8),
        showlegend=False
    ),
    row=2, col=1
)

# Горизонтальная линия для порога Gap = 0.15
fig_weights.add_hline(
    y=0.15, 
    line_dash="dash", 
    line_color="red",
    annotation_text="Порог переобучения (0.15)",
    row=2, col=1
)

# Обновление layout
fig_weights.update_xaxes(title_text="Вес CatBoost", row=2, col=1)
fig_weights.update_yaxes(title_text="Значение", row=1, col=1)
fig_weights.update_yaxes(title_text="Gap", row=2, col=1)

fig_weights.update_layout(
    title={
        'text': 'Оптимизация весов ансамбля XGBoost + CatBoost (с учетом Gap)',
        'x': 0.5,
        'xanchor': 'center'
    },
    template='plotly_white',
    width=900,
    height=700,
    showlegend=True
)

fig_weights.show()


In [40]:
# ============================================================================
# Финальные метрики ансамбля и добавление в результаты
# ============================================================================

# Расчет метрик для оптимального ансамбля (без пост-обработки для честного сравнения)
metrics_ensemble_final = calculate_metrics(
    y_test, y_pred_ensemble_opt,
    f"Ensemble XGB+Cat ({w_cat_best:.0%}/{w_xgb_best:.0%})",
    y_train_true=y_train,
    y_train_pred=y_train_pred_ensemble_opt
)

# Добавляем результат в глобальный список
# Для ансамбля создаём специальную "обёртку"
class EnsembleWrapper:
    """Обёртка для ансамбля, чтобы добавить его в MODEL_RESULTS."""
    def __init__(self, cat_model, xgb_model, w_cat, w_xgb):
        self.cat_model = cat_model
        self.xgb_model = xgb_model
        self.w_cat = w_cat
        self.w_xgb = w_xgb
        self.feature_importances_ = None  # Для совместимости
        
    def predict(self, X):
        # Предполагаем, что X содержит все 16 признаков
        X_base = X[X.columns[:11]]  # Базовые признаки для XGBoost
        X_enhanced = X  # Все признаки для CatBoost
        return get_ensemble_predict(X_base, X_enhanced,
                                    self.cat_model, self.xgb_model,
                                    self.w_cat, self.w_xgb)

ensemble_model = EnsembleWrapper(cat_enhanced, xgb_optimized, w_cat_best, w_xgb_best)

# Добавляем в результаты (Feature-level Diversity: CatBoost на расширенном, XGBoost на базовом)
# Используем ту же структуру, что и функция add_result
MODEL_RESULTS.append({
    'Model': f"Ensemble (Cat+XGB)",
    'Features': f"Mixed (11+16)",
    'R² Train': metrics_ensemble_final['train']['r2'],
    'R² Test': metrics_ensemble_final['test']['r2'],
    'Gap': metrics_ensemble_final['train']['r2'] - metrics_ensemble_final['test']['r2'],
    'RMSE': metrics_ensemble_final['test']['rmse'],
    'MAE': metrics_ensemble_final['test']['mae']
})

print("="*70)
print("АНСАМБЛЬ ДОБАВЛЕН В MODEL_RESULTS")
print("="*70)
print(f"\nВсего моделей для сравнения: {len(MODEL_RESULTS)}")

# Безопасный вывод с проверкой структуры данных
if len(MODEL_RESULTS) == 0:
    print("\n⚠️ ВНИМАНИЕ: MODEL_RESULTS пуст!")
    print("   Необходимо запустить ячейки с обучением моделей перед выполнением этой ячейки.")
else:
    for i, res in enumerate(MODEL_RESULTS, 1):
        try:
            # Проверяем наличие всех необходимых ключей
            model_name = res.get('Model', 'Unknown')
            features = res.get('Features', 'Unknown')
            r2_test = res.get('R² Test', 0.0)
            print(f"  {i}. {model_name} ({features}): R²={r2_test:.4f}")
        except Exception as e:
            print(f"  {i}. ⚠️ Ошибка при выводе модели: {e}")



Ensemble XGB+Cat (30%/70%):
Метрика              Тренировочный набор       Тестовый набор      
-----------------------------------------------------------------
R²                   0.6073                    0.4824              
RMSE                 0.5054                    0.5816              
MAE                  0.3895                    0.4673              

Разница R² (train - test): 0.1249
⚠️ Возможное переобучение (разница > 0.1)
АНСАМБЛЬ ДОБАВЛЕН В MODEL_RESULTS

Всего моделей для сравнения: 7
  1. XGB Baseline (All (11)): R²=0.4280
  2. Cat Baseline (All (11)): R²=0.4272
  3. XGB Optimized (All (11)): R²=0.4522
  4. XGB Enhanced (All (16)): R²=0.4743
  5. CatBoost Optimized (All (11)): R²=0.5157
  6. CatBoost Enhanced (All (16)): R²=0.5122
  7. Ensemble (Cat+XGB) (Mixed (11+16)): R²=0.4824


In [41]:
# ============================================================================
# Анализ согласованности важности признаков между моделями
# ============================================================================

# Важность признаков CatBoost Enhanced (16 признаков)
fi_cat = pd.DataFrame({
    'feature': X_train_plus.columns,
    'importance': cat_enhanced.feature_importances_
}).sort_values('importance', ascending=False)

# Важность признаков XGBoost Optimized (11 признаков)
fi_xgb = pd.DataFrame({
    'feature': X_train_final.columns,
    'importance': xgb_optimized.feature_importances_
}).sort_values('importance', ascending=False)

print("="*70)
print("СРАВНЕНИЕ ВАЖНОСТИ ПРИЗНАКОВ:")
print("="*70)

print("\nТоп-5 признаков CatBoost Enhanced (16 признаков):")
for i, row in fi_cat.head(5).iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

print("\nТоп-5 признаков XGBoost Optimized (базовые 11):")
for i, row in fi_xgb.head(5).iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

# Находим общие топ-признаки
top_cat = set(fi_cat.head(5)['feature'])
top_xgb = set(fi_xgb.head(5)['feature'])
common_top = top_cat.intersection(top_xgb)

print(f"\n✅ Общие топ-признаки (найдены обеими моделями):")
for feat in common_top:
    print(f"   - {feat}")

if len(common_top) >= 2:
    print(f"\n🎯 Высокая согласованность: {len(common_top)} из 5 топ-признаков совпадают!")
    print("   Это означает, что ансамбль нашёл истинные маркеры качества вина.")


СРАВНЕНИЕ ВАЖНОСТИ ПРИЗНАКОВ:

Топ-5 признаков CatBoost Enhanced (16 признаков):
  sulphates: 16.2993
  alc_density_prod: 10.0415
  alcohol: 9.9012
  total sulfur dioxide: 7.4349
  volatile acidity: 7.1678

Топ-5 признаков XGBoost Optimized (базовые 11):
  alcohol: 0.2258
  sulphates: 0.1608
  volatile acidity: 0.1338
  density: 0.0791
  total sulfur dioxide: 0.0751

✅ Общие топ-признаки (найдены обеими моделями):
   - total sulfur dioxide
   - sulphates
   - volatile acidity
   - alcohol

🎯 Высокая согласованность: 4 из 5 топ-признаков совпадают!
   Это означает, что ансамбль нашёл истинные маркеры качества вина.


## Выводы по ансамблю XGBoost + CatBoost

Могу сказать, что эксперимент по объединенную в ансамбль показал видимое улучшение - думаю, если бы у меня был в этом файле доступ к более простым моделям (например случайному лесу из первой части работы), то можно было бы добавить его, привнести еще больше разнообразия и улучшить картину.


# VI. Итоговое сравнение моделей


## 1. Общие итоги и выбор лучших

In [ ]:
comparison_final = MODEL_RESULTS

# Визуализация итогового сравнения
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('R² Test Score (Higher is Better)', 'Gap |Train-Test| (Lower is Better)', 
                    'RMSE (Lower is Better)', 'MAE (Lower is Better)'),
    vertical_spacing=0.15
)

# R2 Test
fig.add_trace(go.Bar(x=comparison_final['Model'], y=comparison_final['R² Test'], 
                     marker_color='steelblue', name='R² Test'), row=1, col=1)

# Gap
fig.add_trace(go.Bar(x=comparison_final['Model'], y=comparison_final['Gap'], 
                     marker_color='orange', name='Gap'), row=1, col=2)

# RMSE
fig.add_trace(go.Bar(x=comparison_final['Model'], y=comparison_final['RMSE'], 
                     marker_color='crimson', name='RMSE'), row=2, col=1)

# MAE
fig.add_trace(go.Bar(x=comparison_final['Model'], y=comparison_final['MAE'], 
                     marker_color='green', name='MAE'), row=2, col=2)

fig.update_layout(height=900, width=1000, title_text="Сравнение эффективности моделей XGBoost и CatBoost", showlegend=False)
fig.update_xaxes(tickangle=45)
fig.show()



Если делать какие то выводы по поводу моделей, то сначала стоит выделить лучшие модели и проанализировать распределение их ошибок: 

- Лучшей среди XGBoosting считаю XGB Optimized - да, она задушена и ее точность на тесте ниже и близка к бейзлайн моделям, зато у нее почти отсутствует переобучение, значит можно доверять ее результатами.
- Среди CatBoost лучшей выберу Catboost Enchanced (модель с дополнительными синтетическими признаками) - в ее случае, моей целью не стояло максимально уменьшить переобучение, мне нужна была более высокая точность, ради проведения эксперимента с ансамблем из двух моделей.
- Ну и ансамбль из этих двух моделей с оптимизированными весами для уменьшения переобучения.

По ошибкам все модели плюс минус равны, разница некритична - все они ошибаются в предсказании оценки примерно на 0.5 пункта, что в целом много конечно, но опять же, учитывая специфику датасета и то, что оценки ставятся реальными людьми, и то что мало данных для обучения, это нельзя однозначно считать плохим результатом

## 2. Анализ распределения ошибок для лучших моделей

Для трёх выбранных моделей проведём детальный анализ распределения ошибок:
- **XGB Optimized** (11 признаков) — стабильная модель с минимальным переобучением
- **CatBoost Enhanced** (16 признаков) — модель с высокой точностью на расширенном наборе признаков
- **Ensemble (Cat+XGB)** — оптимальная комбинация двух моделей с Feature-level Diversity

Анализ включает:
1. Распределение остатков (residuals)
2. График остатков vs истинные значения
3. График предсказаний vs истинные значения
4. Интерпретацию результатов


In [52]:
# ============================================================================
# ПОДГОТОВКА ДАННЫХ ДЛЯ АНАЛИЗА ТРЁХ ЛУЧШИХ МОДЕЛЕЙ
# ============================================================================

# Получаем предсказания для трёх моделей
# 1. XGB Optimized (базовый датасет)
y_pred_xgb_opt_test = xgb_optimized.predict(X_test_final)
y_pred_xgb_opt_train = xgb_optimized.predict(X_train_final)

# 2. CatBoost Enhanced (расширенный датасет)
y_pred_cat_enh_test = cat_enhanced.predict(X_test_plus)
y_pred_cat_enh_train = cat_enhanced.predict(X_train_plus)

# 3. Ensemble (уже вычислены: y_pred_ensemble_opt, y_train_pred_ensemble_opt)

# Создаём DataFrame для сравнения метрик
comparison_best_models = pd.DataFrame([
    {
        'model': 'XGB Optimized',
        'train_r2': metrics_xgb_opt['train']['r2'],
        'test_r2': metrics_xgb_opt['test']['r2'],
        'train_rmse': metrics_xgb_opt['train']['rmse'],
        'test_rmse': metrics_xgb_opt['test']['rmse'],
        'train_mae': metrics_xgb_opt['train']['mae'],
        'test_mae': metrics_xgb_opt['test']['mae'],
        'gap': metrics_xgb_opt['train']['r2'] - metrics_xgb_opt['test']['r2']
    },
    {
        'model': 'CatBoost Enhanced',
        'train_r2': metrics_cat_enhanced['train']['r2'],
        'test_r2': metrics_cat_enhanced['test']['r2'],
        'train_rmse': metrics_cat_enhanced['train']['rmse'],
        'test_rmse': metrics_cat_enhanced['test']['rmse'],
        'train_mae': metrics_cat_enhanced['train']['mae'],
        'test_mae': metrics_cat_enhanced['test']['mae'],
        'gap': metrics_cat_enhanced['train']['r2'] - metrics_cat_enhanced['test']['r2']
    },
    {
        'model': 'Ensemble (Cat+XGB)',
        'train_r2': r2_train_final,
        'test_r2': r2_test_final,
        'train_rmse': np.sqrt(mean_squared_error(y_train, y_train_pred_ensemble_opt)),
        'test_rmse': rmse_final,
        'train_mae': mean_absolute_error(y_train, y_train_pred_ensemble_opt),
        'test_mae': mae_final,
        'gap': gap_final
    }
])

print("="*80)
print("СРАВНЕНИЕ ЛУЧШИХ МОДЕЛЕЙ:")
print("="*80)
print(comparison_best_models.round(4).to_string(index=False))

# Вычисляем остатки (residuals)
residuals_xgb = y_test - y_pred_xgb_opt_test
residuals_cat = y_test - y_pred_cat_enh_test
residuals_ensemble = y_test - y_pred_ensemble_opt

print("\n✅ Данные подготовлены для анализа")


СРАВНЕНИЕ ЛУЧШИХ МОДЕЛЕЙ:
             model  train_r2  test_r2  train_rmse  test_rmse  train_mae  test_mae    gap
     XGB Optimized    0.5082   0.4522      0.5656     0.5983     0.4340    0.4817 0.0560
 CatBoost Enhanced    0.7756   0.5122      0.3821     0.5646     0.2916    0.4447 0.2634
Ensemble (Cat+XGB)    0.6073   0.4824      0.5054     0.5816     0.3895    0.4673 0.1249

✅ Данные подготовлены для анализа


In [54]:
# Анализ распределения остатков (residuals)
print("="*80)
print("АНАЛИЗ РАСПРЕДЕЛЕНИЯ ОСТАТКОВ:")
print("="*80)

fig2 = go.Figure()

fig2.add_trace(go.Histogram(
    x=residuals_xgb,
    name='XGB Optimized',
    marker_color='steelblue',
    opacity=0.7,
    nbinsx=30
))

fig2.add_trace(go.Histogram(
    x=residuals_cat,
    name='CatBoost Enhanced',
    marker_color='crimson',
    opacity=0.7,
    nbinsx=30
))

fig2.add_trace(go.Histogram(
    x=residuals_ensemble,
    name='Ensemble',
    marker_color='green',
    opacity=0.7,
    nbinsx=30
))

fig2.update_layout(
    title={
        'text': 'Распределение остатков для лучших моделей (тестовая выборка)',
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Остаток (y_true - y_pred)',
    yaxis_title='Частота',
    barmode='overlay',
    width=1000,
    height=600,
    template='plotly_white'
)

fig2.show()


АНАЛИЗ РАСПРЕДЕЛЕНИЯ ОСТАТКОВ:


Распределение остатков у всех трех моделей близко к нормальному, ансамбль продемонстрировал более узкое распределение остатков по сравнению с одиночными моделями, что подтверждает снижение вариативности ошибки при усреднении предсказаний.

In [55]:
# График остатков для всех моделей
fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x=y_test,
    y=residuals_xgb,
    mode='markers',
    name='XGB Optimized',
    marker=dict(color='steelblue', size=6, opacity=0.6)
))

fig3.add_trace(go.Scatter(
    x=y_test,
    y=residuals_cat,
    mode='markers',
    name='CatBoost Enhanced',
    marker=dict(color='crimson', size=6, opacity=0.6)
))

fig3.add_trace(go.Scatter(
    x=y_test,
    y=residuals_ensemble,
    mode='markers',
    name='Ensemble',
    marker=dict(color='green', size=6, opacity=0.6)
))

# Горизонтальная линия на нуле
fig3.add_hline(y=0, line_dash="dash", line_color="black", line_width=2)

fig3.update_layout(
    title={
        'text': 'График остатков для лучших моделей',
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Истинные значения (quality)',
    yaxis_title='Остаток (y_true - y_pred)',
    width=1000,
    height=600,
    template='plotly_white'
)

fig3.show()


Если судить по остаткам, то эти модели справляются лучше с предсказанием значений 5 и 6 (самых преобладающих), чем более простые модели, но разница не так велика, к тому же, эти алгоритмы также плохо себя показывают на предсказании редких значений (3, 8), немного лучше на значениях 4 и 7, но тоже видно скос в определенную сторону. Заметны явные выбросы остатков к CatBoost (неудивительно, ведь я не стал сильно бороться с переобучением)

In [56]:
# График предсказаний vs истинных значений
fig4 = go.Figure()

fig4.add_trace(go.Scatter(
    x=y_test,
    y=y_pred_xgb_opt_test,
    mode='markers',
    name='XGB Optimized',
    marker=dict(color='steelblue', size=6, opacity=0.6)
))

fig4.add_trace(go.Scatter(
    x=y_test,
    y=y_pred_cat_enh_test,
    mode='markers',
    name='CatBoost Enhanced',
    marker=dict(color='crimson', size=6, opacity=0.6)
))

fig4.add_trace(go.Scatter(
    x=y_test,
    y=y_pred_ensemble_opt,
    mode='markers',
    name='Ensemble',
    marker=dict(color='green', size=6, opacity=0.6)
))

# Линия идеального предсказания
min_val = min(y_test.min(), y_pred_xgb_opt_test.min(), y_pred_cat_enh_test.min(), y_pred_ensemble_opt.min())
max_val = max(y_test.max(), y_pred_xgb_opt_test.max(), y_pred_cat_enh_test.max(), y_pred_ensemble_opt.max())
fig4.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    name='Идеальное предсказание (y=x)',
    line=dict(color='black', dash='dash', width=2)
))

fig4.update_layout(
    title={
        'text': 'Предсказания vs Истинные значения для лучших моделей',
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Истинные значения (quality)',
    yaxis_title='Предсказанные значения',
    width=1000,
    height=600,
    template='plotly_white'
)

fig4.show()


Модели улавливают линейный тренд, но крайние значения все равно сдвигаются ближе к среднему - проблема связанная с малым количеством пограничных значений подробно описанная в предыдущей работе.

In [59]:
# Интерпретация результатов
print("\n" + "="*80)
print("ИНТЕРПРЕТАЦИЯ РЕЗУЛЬТАТОВ:")
print("="*80)

# Сравнение с базовой моделью XGB Optimized
base_model = comparison_best_models.iloc[0]

print("\nУлучшение относительно XGB Optimized:")

for idx in [1, 2]:
    model = comparison_best_models.iloc[idx]
    
    improvement_r2 = ((model['test_r2'] - base_model['test_r2']) / base_model['test_r2'] * 100)
    improvement_rmse = ((base_model['test_rmse'] - model['test_rmse']) / base_model['test_rmse'] * 100)
    improvement_mae = ((base_model['test_mae'] - model['test_mae']) / base_model['test_mae'] * 100)
    
    print(f"\n{model['model']}:")
    print(f"  Test R²:   {improvement_r2:+.2f}%")
    print(f"  Test RMSE: {improvement_rmse:+.2f}% (уменьшение)")
    print(f"  Test MAE:  {improvement_mae:+.2f}% (уменьшение)")
    print(f"  Gap:       {model['gap']:.4f} (XGB: {base_model['gap']:.4f})")


ИНТЕРПРЕТАЦИЯ РЕЗУЛЬТАТОВ:

Улучшение относительно XGB Optimized:

CatBoost Enhanced:
  Test R²:   +13.27%
  Test RMSE: +5.64% (уменьшение)
  Test MAE:  +7.68% (уменьшение)
  Gap:       0.2634 (XGB: 0.0560)

Ensemble (Cat+XGB):
  Test R²:   +6.69%
  Test RMSE: +2.80% (уменьшение)
  Test MAE:  +2.99% (уменьшение)
  Gap:       0.1249 (XGB: 0.0560)


In [61]:
# Анализ остатков
print("\n" + "="*80)
print("АНАЛИЗ ОСТАТКОВ:")
print("="*80)

for name, residuals in [('XGB Optimized', residuals_xgb), 
                         ('CatBoost Enhanced', residuals_cat), 
                         ('Ensemble', residuals_ensemble)]:
    print(f"\n{name}:")
    print(f"  Среднее остатков: {np.mean(residuals):.4f}")
    print(f"  Стд. откл.:       {np.std(residuals):.4f}")
    print(f"  Min/Max:          {np.min(residuals):.4f} / {np.max(residuals):.4f}")

print("\n✅ Анализ распределения ошибок завершён")



АНАЛИЗ ОСТАТКОВ:

XGB Optimized:
  Среднее остатков: 0.0238
  Стд. откл.:       0.5979
  Min/Max:          -2.3982 / 1.6286

CatBoost Enhanced:
  Среднее остатков: 0.0287
  Стд. откл.:       0.5639
  Min/Max:          -2.1552 / 1.4850

Ensemble:
  Среднее остатков: 0.0252
  Стд. откл.:       0.5810
  Min/Max:          -2.3253 / 1.5804

✅ Анализ распределения ошибок завершён


### Выводы по анализу распределения ошибок

Анализ остатков подтверждает неплохое качество и адекватность обученных моделей: средние значения ошибок (0.023–0.028) максимально близки к нулю, что свидетельствует об отсутствии систематического смещения и «честности» прогнозов. CatBoost Enhanced демонстрирует наименьшее стандартное отклонение (0.5639), подтверждая свою лидирующую точность, в то время как ансамбль успешно балансирует между агрессивной точностью CatBoost и консервативной стабильностью XGBoost.

Размах ошибок (от -2.4 до +1.6) указывает на то, что модели чаще склонны переоценивать вина низкого качества, чем недооценивать премиальные образцы, что обусловлено дефицитом примеров с экстремальными оценками в датасете. Тем не менее, стандартное отклонение менее 0.6 балла является отличным результатом для данной задачи: при дискретном шаге целевой переменной в 1.0 балл такая погрешность в большинстве случаев позволяет получить верную оценку после округления, делая итоговый ансамбль неплохим инструментом предсказания, как сбалансированное решение.


# VII. Выводы


Сравнение лучшей линейной и нелинейных моделей:

                               model  train_r2  test_r2  train_mse  test_mse  train_mae  test_mae
               Polynomial LR (deg=2)  0.428755 0.415680   0.371648  0.381857   0.471844  0.494774
               SVR (C=1, kernel=rbf)  0.526573 0.462324   0.308008  0.351374   0.384531  0.453528
               DecisionTreeRegressor  0.489849 0.365276   0.331900  0.414796   0.428556  0.502601




Сравнение моделей из первой части бустингов:

                     Model      Features  R² Train  R² Test    Gap   RMSE    MAE
               RF Baseline      All (11)    0.9265   0.5404 0.3861 0.5481 0.4202
               GB Baseline      All (11)    0.6173   0.4414 0.1759 0.6042 0.4856
        RF Optimized (All)      All (11)    0.6964   0.5056 0.1908 0.5684 0.4525
    RF Optimized (Reduced)         Top 5    0.6490   0.4659 0.1831 0.5908 0.4698
    RF Enhanced  New Feat       All (16)    0.6414   0.4813 0.1601 0.5822 0.4688
        GB Optimized (All)      All (11)    0.5917   0.4810 0.1107 0.5824 0.4699
       GB Optimized (Poly) Enhanced (14)    0.6397   0.4984 0.1413 0.5725 0.4565


Сравнение моделей второй части бустингов:

             model  train_r2  test_r2  train_rmse  test_rmse  train_mae  test_mae    gap
     XGB Optimized    0.5082   0.4522      0.5656     0.5983     0.4340    0.4817 0.0560
    CatBoost Enh      0.7756   0.5122      0.3821     0.5646     0.2916    0.4447 0.2634
    Ensemble          0.6073   0.4824      0.5054     0.5816     0.3895    0.4673 0.1249

Я привел таблицы, в которых можно увидеть результаты моделей из прошлой работы, в которой использовались линейные и нелинейные модели регресси, из первой части, где применялись бустинги из sklearn и из этой части, где применялись сложные бустинги. Анализируя результаты я делаю следующие выводы:

1. Данную задачу предсказания дискретных оценок вина нужно определенно пробовать решать через классификаторы - оценки целые, значит дискретные, учитывая специфику неравномерного количества оценок, в случае классификации можно попробовать сбалансировать классы. Решая регрессию я хоть и получаю вменяемый результат с неплохой ошибкой для целых оценок, но полбалла это много.

2. Если опустить глаза на, возможно неверный подход изначально, то можно сказать, что для данного датасета идея использовать более сложные алгоритмы не имеет особого смысла - CatBoost и XGBoosting слишком сложные для этих данных, это как стрелять из пушки по воробьям - метод опорных векторов справляется ничуть не хуже, если предположить, что мы ищем баланс между точностью и переобучением. Конечно, если хочется выбить максимальную метрику на тесте, то тут бустинги отлично подходят, но какой в этом смысл, если потом приходится их душить, чтобы ликвидировать оверфиттинг.

3. Опыт использование ансамблевых моделей весьма интересный - есть мнение, что добавь я туда еще модель RF например, то результат мог быть интересней - но в целом, это хорошая опция поиска баланса.

4. GridSearch пытается выбить максимальную метрику на трейне, потом приходится самому руками искать нужжные гиперпараметры, чтобы добиваться вменяемого результата - наверно я тут не разобрался с целевой метрикой.

5. Мне казалось, что добавление новых признаков поможет лучше объяснить модели хитрости химического взаимодействия, но на это стоило делать больший упор возможно на более простых моделях - это дает пользу, учитывая, что я в ансамбле использую CatBoost с добавленными признакми, но можно было бы обойтись и без них.

В целом, результат считаю все равно хорошим - для задачи, в которой целевая переменная является субъективной человеческой оценкой как буд-то тяжело достичь результата лучше, ведь машина не может предсказать все суждения, которые придут в голову сомелье. В выводах к прошлой работе писал о своей уверенности, что более сложная модель справится с поиском сложных закономерностей - возможно так и есть, если бы данных изначально было бы больше.